In [1]:
"""
ASL Sign Language Converter
Combines ResNet (Letter Model) + LSTM (Word Model)
With debounce, buffer, TTS, and clean UI
"""

import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp
from collections import deque
import pyttsx3
import time
import threading
import tkinter as tk
import customtkinter as ctk
from PIL import Image, ImageTk
# ─────────────────────────────────────────────
#  HAND SKELETON HELPERS  (from your main.ipynb)
# ─────────────────────────────────────────────

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
    (5, 9), (9, 13), (13, 17)
]

FINGER_COLORS = {
    (0, 1): (0, 255, 255),  (1, 2): (0, 255, 255),  (2, 3): (0, 255, 255),  (3, 4): (0, 255, 255),
    (0, 5): (255, 80, 80),  (5, 6): (255, 80, 80),  (6, 7): (255, 80, 80),  (7, 8): (255, 80, 80),
    (0, 9): (80, 255, 80),  (9, 10): (80, 255, 80), (10, 11): (80, 255, 80), (11, 12): (80, 255, 80),
    (0, 13): (80, 80, 255), (13, 14): (80, 80, 255), (14, 15): (80, 80, 255), (15, 16): (80, 80, 255),
    (0, 17): (255, 80, 255), (17, 18): (255, 80, 255), (18, 19): (255, 80, 255), (19, 20): (255, 80, 255),
    (5, 9): (220, 220, 220), (9, 13): (220, 220, 220), (13, 17): (220, 220, 220)
}

def normalize_landmarks(landmarks, img_w, img_h):
    points = np.array([[lm.x * img_w, lm.y * img_h] for lm in landmarks])
    mn = points.min(axis=0); mx = points.max(axis=0)
    cx = (mn[0] + mx[0]) / 2; cy = (mn[1] + mx[1]) / 2
    dim = max(mx[0] - mn[0], mx[1] - mn[1])
    scale = (min(img_w, img_h) * 0.8) / dim if dim > 0 else 1
    return [(int((p[0] - cx) * scale + img_w / 2),
             int((p[1] - cy) * scale + img_h / 2)) for p in points]

def draw_hand_skeleton(frame, pts):
    for c in HAND_CONNECTIONS:
        cv2.line(frame, pts[c[0]], pts[c[1]], FINGER_COLORS[c], 3)
    for i, p in enumerate(pts):
        color = (0, 0, 255) if i in [4, 8, 12, 16, 20] else (255, 0, 255)
        cv2.circle(frame, p, 3, color, -1)
        cv2.circle(frame, p, 3, (255, 255, 255), 1)
    return frame

# ─────────────────────────────────────────────
#  KEYPOINT EXTRACTION  (from FinalLSTMMODEL)
# ─────────────────────────────────────────────

mp_holistic = mp.solutions.holistic
mp_hands_mod = mp.solutions.hands

def extract_keypoints(results):
    pose = (np.array([[r.x, r.y, r.z, r.visibility] for r in results.pose_landmarks.landmark]).flatten()
            if results.pose_landmarks else np.zeros(132))
    lh   = (np.array([[r.x, r.y, r.z] for r in results.left_hand_landmarks.landmark]).flatten()
            if results.left_hand_landmarks else np.zeros(63))
    rh   = (np.array([[r.x, r.y, r.z] for r in results.right_hand_landmarks.landmark]).flatten()
            if results.right_hand_landmarks else np.zeros(63))
    return np.concatenate([pose, lh, rh])


# ─────────────────────────────────────────────
#  MAIN APPLICATION
# ─────────────────────────────────────────────

class ASLApp:
    # ── labels ──
    LETTER_LABELS = ['A','B','C','D','E','F','G','H','I','J','K','L','M',
                     'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
                     'del','ok','space']
    WORD_LABELS   = ['hello','help','meet','nice','nothing','please','thanks']

    DEBOUNCE_SEC  = 1.5   # seconds to hold a letter before confirming
    WORD_STAB_FRM = 8     # consecutive frames for word confirmation
    CONF_THRESH   = 0.70  # minimum confidence

    def __init__(self, root: ctk.CTk):
        self.root = root
        self.root.title("ASL Sign Language Converter")
        self.root.geometry("1180x780")
        self.root.resizable(False, False)

        # ── state ──
        self.mode          = "letter"   # "letter" | "word"
        self.buffer        = ""
        self.running       = True

        # letter debounce
        self.pending_letter   = None
        self.pending_since    = None

        # word stability
        self.word_sequence    = deque(maxlen=30)
        self.last_word_pred   = None
        self.word_stab_count  = 0
        self.last_confirmed_word = None

        # display
        self.cur_prediction   = ""
        self.cur_confidence   = 0.0
        self.skeleton_frame   = np.zeros((224, 224, 3), dtype=np.uint8)

        # TTS
        self.tts_engine = pyttsx3.init()
        self.tts_engine.setProperty('rate', 150)

        # models
        self.letter_model  = None
        self.word_model    = None
        self.holistic      = None
        self.hand_detector = None

        self._build_ui()
        self._load_models_async()

        # camera
        self.cap = cv2.VideoCapture(0)
        self._camera_loop()

    # ──────────────────────────────────────────
    #  UI BUILDER
    # ──────────────────────────────────────────

    def _build_ui(self):
        BG      = "#0d0d0f"
        PANEL   = "#16161a"
        ACCENT  = "#00e5ff"
        ACCENT2 = "#ff3d71"
        TEXT    = "#e8e8f0"
        SUBTEXT = "#6b6b80"
        BTN_H   = "#1a1a22"

        self.root.configure(fg_color=BG)

        # ── top bar ──
        topbar = ctk.CTkFrame(self.root, fg_color=PANEL, height=54, corner_radius=0)
        topbar.pack(fill="x", side="top")
        ctk.CTkLabel(topbar, text="◈  ASL CONVERTER",
                     font=("Courier New", 17, "bold"), text_color=ACCENT).pack(side="left", padx=20, pady=14)
        self.status_dot = ctk.CTkLabel(topbar, text="● LOADING",
                                       font=("Courier New", 12), text_color="#ffaa00")
        self.status_dot.pack(side="right", padx=20)

        # ── main body ──
        body = ctk.CTkFrame(self.root, fg_color=BG)
        body.pack(fill="both", expand=True, padx=0, pady=0)

        # left – camera
        left = ctk.CTkFrame(body, fg_color=PANEL, corner_radius=14)
        left.pack(side="left", padx=(14, 7), pady=14, fill="both", expand=True)

        ctk.CTkLabel(left, text="LIVE FEED", font=("Courier New", 10, "bold"),
                     text_color=SUBTEXT).pack(anchor="nw", padx=14, pady=(10, 0))
        self.video_lbl = ctk.CTkLabel(left, text="", fg_color="#0a0a0c", corner_radius=10)
        self.video_lbl.pack(padx=10, pady=6)

        ctk.CTkLabel(left, text="SKELETON", font=("Courier New", 10, "bold"),
                     text_color=SUBTEXT).pack(anchor="nw", padx=14, pady=(4, 0))
        self.skel_lbl = ctk.CTkLabel(left, text="", fg_color="#0a0a0c", corner_radius=10)
        self.skel_lbl.pack(padx=10, pady=(4, 10))

        # right panel
        right = ctk.CTkFrame(body, fg_color=BG, width=360)
        right.pack(side="right", padx=(7, 14), pady=14, fill="y")
        right.pack_propagate(False)

        # mode toggle
        mode_card = ctk.CTkFrame(right, fg_color=PANEL, corner_radius=12)
        mode_card.pack(fill="x", pady=(0, 10))
        ctk.CTkLabel(mode_card, text="MODE", font=("Courier New", 10, "bold"),
                     text_color=SUBTEXT).pack(anchor="nw", padx=14, pady=(10, 4))

        self.mode_var = ctk.StringVar(value="letter")
        toggle_row = ctk.CTkFrame(mode_card, fg_color="transparent")
        toggle_row.pack(padx=10, pady=(0, 12))

        self.btn_letter = ctk.CTkButton(
            toggle_row, text="✦  LETTER", width=155, height=40,
            fg_color=ACCENT, hover_color="#00b8d4", text_color="#0d0d0f",
            font=("Courier New", 13, "bold"), corner_radius=8,
            command=lambda: self._set_mode("letter"))
        self.btn_letter.pack(side="left", padx=(0, 6))

        self.btn_word = ctk.CTkButton(
            toggle_row, text="✦  WORD", width=155, height=40,
            fg_color=BTN_H, hover_color="#222230", text_color=SUBTEXT,
            font=("Courier New", 13, "bold"), corner_radius=8,
            command=lambda: self._set_mode("word"))
        self.btn_word.pack(side="left")

        # prediction card
        pred_card = ctk.CTkFrame(right, fg_color=PANEL, corner_radius=12)
        pred_card.pack(fill="x", pady=(0, 10))
        ctk.CTkLabel(pred_card, text="PREDICTION", font=("Courier New", 10, "bold"),
                     text_color=SUBTEXT).pack(anchor="nw", padx=14, pady=(10, 0))

        self.pred_lbl = ctk.CTkLabel(pred_card, text="—",
                                     font=("Courier New", 52, "bold"), text_color=ACCENT)
        self.pred_lbl.pack(pady=(4, 0))

        self.conf_bar = ctk.CTkProgressBar(pred_card, width=300, height=6,
                                            fg_color="#1e1e28", progress_color=ACCENT,
                                            corner_radius=4)
        self.conf_bar.set(0)
        self.conf_bar.pack(pady=(4, 6))

        self.conf_lbl = ctk.CTkLabel(pred_card, text="confidence: 0%",
                                     font=("Courier New", 11), text_color=SUBTEXT)
        self.conf_lbl.pack(pady=(0, 10))

        # debounce indicator (letter mode only)
        self.debounce_card = ctk.CTkFrame(right, fg_color=PANEL, corner_radius=12)
        self.debounce_card.pack(fill="x", pady=(0, 10))
        ctk.CTkLabel(self.debounce_card, text="CONFIRMING",
                     font=("Courier New", 10, "bold"), text_color=SUBTEXT).pack(anchor="nw", padx=14, pady=(10, 0))
        self.debounce_bar = ctk.CTkProgressBar(self.debounce_card, width=300, height=8,
                                                fg_color="#1e1e28", progress_color=ACCENT2,
                                                corner_radius=4)
        self.debounce_bar.set(0)
        self.debounce_bar.pack(pady=(4, 4))
        self.debounce_lbl = ctk.CTkLabel(self.debounce_card, text="hold sign to confirm…",
                                          font=("Courier New", 11), text_color=SUBTEXT)
        self.debounce_lbl.pack(pady=(0, 10))

        # buffer / sentence
        buf_card = ctk.CTkFrame(right, fg_color=PANEL, corner_radius=12)
        buf_card.pack(fill="x", pady=(0, 10))
        ctk.CTkLabel(buf_card, text="SENTENCE BUFFER", font=("Courier New", 10, "bold"),
                     text_color=SUBTEXT).pack(anchor="nw", padx=14, pady=(10, 4))
        self.buffer_lbl = ctk.CTkLabel(buf_card, text="",
                                        font=("Courier New", 22, "bold"), text_color=TEXT,
                                        wraplength=320, justify="left")
        self.buffer_lbl.pack(padx=14, pady=(0, 12), anchor="w")

        # action buttons
        btn_card = ctk.CTkFrame(right, fg_color="transparent")
        btn_card.pack(fill="x", pady=(0, 0))

        self.btn_speak = ctk.CTkButton(
            btn_card, text="▶  SPEAK", height=44, corner_radius=10,
            fg_color=ACCENT, hover_color="#00b8d4", text_color="#0d0d0f",
            font=("Courier New", 14, "bold"), command=self._speak)
        self.btn_speak.pack(fill="x", pady=(0, 8))

        self.btn_del = ctk.CTkButton(
            btn_card, text="⌫  DELETE LAST", height=38, corner_radius=10,
            fg_color=BTN_H, hover_color="#2a1a1a", text_color=ACCENT2,
            font=("Courier New", 12, "bold"), command=self._delete_last)
        self.btn_del.pack(fill="x", pady=(0, 6))

        self.btn_clear = ctk.CTkButton(
            btn_card, text="✕  CLEAR ALL", height=38, corner_radius=10,
            fg_color=BTN_H, hover_color="#2a1a1a", text_color=SUBTEXT,
            font=("Courier New", 12, "bold"), command=self._clear)
        self.btn_clear.pack(fill="x")

    # ──────────────────────────────────────────
    #  MODE TOGGLE
    # ──────────────────────────────────────────

    def _set_mode(self, mode):
        self.mode = mode
        ACCENT = "#00e5ff"; SUBTEXT = "#6b6b80"; BTN_H = "#1a1a22"
        if mode == "letter":
            self.btn_letter.configure(fg_color=ACCENT, text_color="#0d0d0f")
            self.btn_word.configure(fg_color=BTN_H, text_color=SUBTEXT)
            self.debounce_card.pack(fill="x", pady=(0, 10))
        else:
            self.btn_word.configure(fg_color=ACCENT, text_color="#0d0d0f")
            self.btn_letter.configure(fg_color=BTN_H, text_color=SUBTEXT)
            self.debounce_card.pack_forget()
        # reset state
        self.pending_letter  = None
        self.pending_since   = None
        self.word_sequence.clear()
        self.word_stab_count = 0
        self.last_word_pred  = None
        self.last_confirmed_word = None

    # ──────────────────────────────────────────
    #  LOAD MODELS (background thread)
    # ──────────────────────────────────────────

    def _load_models_async(self):
        threading.Thread(target=self._load_models, daemon=True).start()

    def _load_models(self):
        try:
            self.letter_model = tf.keras.models.load_model('/home/maries/Development/Project_OWSLT/models/letter_modelV2.keras')
            self.root.after(0, lambda: self.status_dot.configure(
                text="● LETTER OK  WORD LOADING", text_color="#ffaa00"))
        except Exception as e:
            self.root.after(0, lambda: self.status_dot.configure(
                text=f"⚠ LETTER FAIL: {e}", text_color="#ff3d71"))

        try:
            self.word_model = tf.keras.models.load_model('/home/maries/Development/Project_OWSLT/models/word_modelV2.keras')
            self.root.after(0, lambda: self.status_dot.configure(
                text="● READY", text_color="#00e5ff"))
        except Exception as e:
            self.root.after(0, lambda: self.status_dot.configure(
                text=f"⚠ WORD FAIL: {e}", text_color="#ff3d71"))

        # init mediapipe
        self.holistic      = mp_holistic.Holistic(min_detection_confidence=0.5,
                                                   min_tracking_confidence=0.5)
        self.hand_detector = mp_hands_mod.Hands(min_detection_confidence=0.5,
                                                 min_tracking_confidence=0.5,
                                                 max_num_hands=1)

    # ──────────────────────────────────────────
    #  CAMERA LOOP
    # ──────────────────────────────────────────

    def _camera_loop(self):
        if not self.running:
            return

        ret, frame = self.cap.read()
        if ret:
            frame = cv2.flip(frame, 1)
            skel  = np.zeros((224, 224, 3), dtype=np.uint8)

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            if self.mode == "letter":
                self._process_letter(rgb, frame, skel)
            else:
                self._process_word(rgb, frame)

            # draw UI overlay on frame
            self._draw_overlay(frame)

            # update video label
            h, w = frame.shape[:2]
            scale = 560 / w
            disp = cv2.resize(rgb, (560, int(h * scale)))
            img  = Image.fromarray(disp)
            cimg = ctk.CTkImage(light_image=img, dark_image=img, size=(560, int(h * scale)))
            self.video_lbl.configure(image=cimg)
            self.video_lbl._image = cimg

            # skeleton
            skel_rgb = cv2.cvtColor(self.skeleton_frame, cv2.COLOR_BGR2RGB)
            skel_img = Image.fromarray(skel_rgb)
            cskel    = ctk.CTkImage(light_image=skel_img, dark_image=skel_img, size=(140, 140))
            self.skel_lbl.configure(image=cskel)
            self.skel_lbl._image = cskel

        self.root.after(15, self._camera_loop)

    # ──────────────────────────────────────────
    #  LETTER PROCESSING  (ResNet + debounce)
    # ──────────────────────────────────────────

    def _process_letter(self, rgb, bgr, skel):
        if self.hand_detector is None or self.letter_model is None:
            return

        result = self.hand_detector.process(rgb)
        if result.multi_hand_landmarks:
            for hand in result.multi_hand_landmarks:
                pts = normalize_landmarks(hand.landmark, 224, 224)
                draw_hand_skeleton(skel, pts)
            self.skeleton_frame = skel

            inp = tf.keras.applications.resnet50.preprocess_input(
                np.expand_dims(skel, axis=0).astype(np.float32))
            pred = self.letter_model.predict(inp, verbose=0)[0]
            idx  = int(np.argmax(pred))
            conf = float(pred[idx])
            label = self.LETTER_LABELS[idx]

            self.cur_prediction  = label
            self.cur_confidence  = conf

            self._update_pred_ui(label, conf)

            if conf >= self.CONF_THRESH and label not in ('ok',):
                now = time.time()
                if self.pending_letter != label:
                    self.pending_letter = label
                    self.pending_since  = now

                elapsed = now - (self.pending_since or now)
                progress = min(elapsed / self.DEBOUNCE_SEC, 1.0)
                self.debounce_bar.set(progress)
                self.debounce_lbl.configure(
                    text=f'confirming "{label}"… {int(progress*100)}%')

                if elapsed >= self.DEBOUNCE_SEC:
                    self._confirm_letter(label)
                    self.pending_letter = None
                    self.pending_since  = None
                    self.debounce_bar.set(0)
                    self.debounce_lbl.configure(text="hold sign to confirm…")
            else:
                self.pending_letter = None
                self.pending_since  = None
                self.debounce_bar.set(0)

    def _confirm_letter(self, letter):
        if letter == 'del':
            self.buffer = self.buffer[:-1]
        elif letter == 'space':
            self.buffer += ' '
        else:
            self.buffer += letter
        self._update_buffer_ui()

    # ──────────────────────────────────────────
    #  WORD PROCESSING  (LSTM + stability)
    # ──────────────────────────────────────────

    def _process_word(self, rgb, bgr):
        if self.holistic is None or self.word_model is None:
            return

        img_rgb = rgb.copy()
        img_rgb.flags.writeable = False
        result  = self.holistic.process(img_rgb)

        # draw skeleton on blank
        skel = np.zeros((224, 224, 3), dtype=np.uint8)
        if result.right_hand_landmarks:
            for lm in result.right_hand_landmarks.landmark:
                pass  # keypoints handled below
        self.skeleton_frame = skel

        kp = extract_keypoints(result)
        self.word_sequence.append(kp)

        if len(self.word_sequence) == 30:
            seq = np.expand_dims(np.array(self.word_sequence), axis=0)
            pred = self.word_model.predict(seq, verbose=0)[0]
            idx  = int(np.argmax(pred))
            conf = float(pred[idx])
            label = self.WORD_LABELS[idx]

            self._update_pred_ui(label, conf)

            if conf >= self.CONF_THRESH:
                if idx == self.last_word_pred:
                    self.word_stab_count += 1
                else:
                    self.last_word_pred  = idx
                    self.word_stab_count = 1

                if (self.word_stab_count >= self.WORD_STAB_FRM
                        and label != 'nothing'
                        and label != self.last_confirmed_word):
                    self.buffer += label + ' '
                    self.last_confirmed_word = label
                    self.word_stab_count = 0
                    self._update_buffer_ui()
            else:
                self.last_word_pred  = None
                self.word_stab_count = 0

    # ──────────────────────────────────────────
    #  OVERLAY ON CAMERA FRAME
    # ──────────────────────────────────────────

    def _draw_overlay(self, frame):
        h, w = frame.shape[:2]
        # mode badge
        tag = "LETTER" if self.mode == "letter" else "WORD"
        col = (0, 229, 255) if self.mode == "letter" else (255, 61, 113)
        cv2.rectangle(frame, (w - 120, 0), (w, 32), (13, 13, 15), -1)
        cv2.putText(frame, tag, (w - 110, 22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.7, col, 1, cv2.LINE_AA)

        # guide crosshair
        cx, cy = w // 2, h // 2
        cv2.line(frame, (cx - 20, cy), (cx + 20, cy), (40, 40, 50), 1)
        cv2.line(frame, (cx, cy - 20), (cx, cy + 20), (40, 40, 50), 1)

    # ──────────────────────────────────────────
    #  UI UPDATES
    # ──────────────────────────────────────────

    def _update_pred_ui(self, label, conf):
        self.pred_lbl.configure(text=label.upper())
        self.conf_bar.set(conf)
        self.conf_lbl.configure(text=f"confidence: {int(conf*100)}%")

    def _update_buffer_ui(self):
        self.buffer_lbl.configure(text=self.buffer if self.buffer else "")

    # ──────────────────────────────────────────
    #  ACTIONS
    # ──────────────────────────────────────────

    def _speak(self):
        text = self.buffer.strip()
        if text:
            threading.Thread(target=self._tts_run, args=(text,), daemon=True).start()

    def _tts_run(self, text):
        self.tts_engine.say(text)
        self.tts_engine.runAndWait()

    def _delete_last(self):
        self.buffer = self.buffer.rstrip()
        if ' ' in self.buffer:
            self.buffer = self.buffer.rsplit(' ', 1)[0] + ' '
        else:
            self.buffer = self.buffer[:-1]
        self._update_buffer_ui()

    def _clear(self):
        self.buffer = ""
        self._update_buffer_ui()
        self.pending_letter = None
        self.pending_since  = None
        self.debounce_bar.set(0)

    # ──────────────────────────────────────────
    #  CLOSE
    # ──────────────────────────────────────────

    def on_close(self):
        self.running = False
        self.cap.release()
        if self.holistic:
            self.holistic.close()
        if self.hand_detector:
            self.hand_detector.close()
        self.root.destroy()


# ──────────────────────────────────────────────
#  ENTRY POINT
# ──────────────────────────────────────────────

if __name__ == "__main__":
    ctk.set_appearance_mode("dark")
    ctk.set_default_color_theme("blue")

    root = ctk.CTk()
    app  = ASLApp(root)
    root.protocol("WM_DELETE_WINDOW", app.on_close)
    root.mainloop()

2026-03-07 08:53:14.470340: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 08:53:14.608990: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-07 08:53:14.609038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-07 08:53:14.613689: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-07 08:53:14.631704: I tensorflow/core/platform/cpu_feature_guar

/home/maries/Development/Project_OWSLT/main scripts


In [2]:
"""
╔══════════════════════════════════════════════════════╗
║        ASL Sign Language Translator  —  Tkinter UI   ║
║                                                      ║
║  • Looks up whole WORD signs as GIFs from signasl.org║
║  • Falls back to letter-by-letter fingerspelling     ║
║    using hand-photo PNGs from a public GitHub repo   ║
║  • Animated GIF playback in the video canvas         ║
║                                                      ║
║  Requirements (install once):                        ║
║    pip install pillow requests                       ║
╚══════════════════════════════════════════════════════╝
"""

import tkinter as tk
from tkinter import ttk, font as tkfont
import threading
import requests
import os
import re
import time
from io import BytesIO
from PIL import Image, ImageTk, ImageDraw, ImageFont, ImageSequence

# ── Cache folder ──────────────────────────────────────────────────────────────
CACHE_DIR = os.path.join(os.path.expanduser("~"), ".asl_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Letter image source (MIT-licensed Kaggle test set via a public mirror) ────
# These are real hand-photo PNG images of each ASL letter A-Z
# Source: https://github.com/AkashNagaraj/ASL-Recognition (public Kaggle images)
# Fallback mirror if first fails.
LETTER_SOURCES = [
    # Primary: clean white-background hand photos, MIT-friendly Kaggle test set
    "https://raw.githubusercontent.com/Temitope-py/sign-language-detector/main/data/test/{letter}/{letter}1.jpg",
    # Second mirror
    "https://raw.githubusercontent.com/AkashNagaraj/ASL-Recognition/master/Test/{LETTER}/{LETTER}_test1.jpg",
]

# signasl.org has embedded mp4/gif per word — we scrape the gif link
SIGNASL_SEARCH = "https://www.signasl.org/sign/{word}"

CANVAS_W, CANVAS_H = 480, 380

# ── Colour palette ─────────────────────────────────────────────────────────────
BG       = "#0d1117"
SURFACE  = "#161b22"
CARD     = "#21262d"
BORDER   = "#30363d"
ACCENT   = "#58a6ff"
ACCENT2  = "#f78166"
GREEN    = "#3fb950"
MUTED    = "#8b949e"
TEXT     = "#e6edf3"
WHITE    = "#ffffff"

# ═══════════════════════════════════════════════════════════════════════════════
#  Image fetching helpers
# ═══════════════════════════════════════════════════════════════════════════════

def cache_path(key: str, ext: str) -> str:
    safe = re.sub(r'[^a-zA-Z0-9_\-]', '_', key)
    return os.path.join(CACHE_DIR, f"{safe}.{ext}")


def fetch_bytes(url: str, timeout=8) -> bytes | None:
    try:
        headers = {"User-Agent": "Mozilla/5.0 (ASL-Translator-App/1.0)"}
        r = requests.get(url, timeout=timeout, headers=headers)
        if r.status_code == 200:
            return r.content
    except Exception:
        pass
    return None


def get_letter_image(letter: str) -> Image.Image | None:
    """Download / cache a real hand-photo for the given ASL letter."""
    letter = letter.upper()
    cp = cache_path(f"letter_{letter}", "png")
    if os.path.exists(cp):
        try:
            return Image.open(cp).convert("RGBA")
        except Exception:
            pass

    urls = [
        f"https://raw.githubusercontent.com/Temitope-py/sign-language-detector/main/data/test/{letter}/{letter}1.jpg",
        f"https://raw.githubusercontent.com/Temitope-py/sign-language-detector/main/data/test/{letter.lower()}/{letter.lower()}1.jpg",
        f"https://raw.githubusercontent.com/AkashNagaraj/ASL-Recognition/master/Test/{letter}/{letter}_test1.jpg",
        # broad fallback: Wikipedia commons ASL chart crop is CC0
        f"https://upload.wikimedia.org/wikipedia/commons/thumb/9/9a/Sign_language_{letter.lower()}.svg/240px-Sign_language_{letter.lower()}.svg.png",
    ]

    for url in urls:
        data = fetch_bytes(url)
        if data:
            try:
                img = Image.open(BytesIO(data)).convert("RGBA")
                img.save(cp)
                return img
            except Exception:
                continue
    return None


def get_word_gif_url(word: str) -> str | None:
    """Scrape signasl.org for a direct GIF/video URL for a whole word."""
    word = word.lower().strip()
    url  = SIGNASL_SEARCH.format(word=word)
    data = fetch_bytes(url, timeout=10)
    if not data:
        return None
    html = data.decode("utf-8", errors="ignore")

    # Look for <source src="...gif"> or <img src="...gif">
    for pattern in [
        r'<source[^>]+src=["\']([^"\']+\.gif)["\']',
        r'<img[^>]+src=["\']([^"\']+\.gif)["\']',
        r'(https?://[^\s"\'<>]+\.gif)',
    ]:
        m = re.search(pattern, html, re.IGNORECASE)
        if m:
            return m.group(1)

    # Try mp4 → we'll show first frame as still image
    for pattern in [
        r'<source[^>]+src=["\']([^"\']+\.mp4)["\']',
        r'(https?://[^\s"\'<>]+\.mp4)',
    ]:
        m = re.search(pattern, html, re.IGNORECASE)
        if m:
            return m.group(1)

    return None


def get_word_gif_frames(word: str) -> list[Image.Image] | None:
    """Return list of PIL frames for word GIF, or None if not found."""
    word = word.lower().strip()
    cp   = cache_path(f"word_{word}", "gif")

    gif_url = None
    if not os.path.exists(cp):
        gif_url = get_word_gif_url(word)
        if not gif_url:
            return None
        data = fetch_bytes(gif_url, timeout=15)
        if not data:
            return None
        with open(cp, "wb") as f:
            f.write(data)

    try:
        img    = Image.open(cp)
        frames = [frame.copy().convert("RGBA") for frame in ImageSequence.Iterator(img)]
        return frames if frames else None
    except Exception:
        if os.path.exists(cp):
            os.remove(cp)
        return None


def make_placeholder_frame(text: str, sub: str = "", bg=(20, 25, 35)) -> Image.Image:
    """Create a simple placeholder frame when no image is available."""
    img  = Image.new("RGBA", (CANVAS_W, CANVAS_H), bg + (255,))
    draw = ImageDraw.Draw(img)

    # big letter/word
    try:
        fnt_big = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 120)
        fnt_sub = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 22)
    except Exception:
        fnt_big = ImageFont.load_default()
        fnt_sub = fnt_big

    bbox = draw.textbbox((0, 0), text, font=fnt_big)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    draw.text(((CANVAS_W - tw) // 2, (CANVAS_H - th) // 2 - 30), text,
              fill=(88, 166, 255, 230), font=fnt_big)

    if sub:
        bbox2  = draw.textbbox((0, 0), sub, font=fnt_sub)
        tw2    = bbox2[2] - bbox2[0]
        draw.text(((CANVAS_W - tw2) // 2, CANVAS_H - 60), sub,
                  fill=(139, 148, 158, 200), font=fnt_sub)
    return img


def fit_image(img: Image.Image, w: int, h: int) -> Image.Image:
    img.thumbnail((w, h), Image.LANCZOS)
    canvas = Image.new("RGBA", (w, h), (13, 17, 23, 255))
    x = (w - img.width) // 2
    y = (h - img.height) // 2
    canvas.paste(img, (x, y), img if img.mode == "RGBA" else None)
    return canvas


# ═══════════════════════════════════════════════════════════════════════════════
#  Sequence builder  →  list of (label, source_hint, frames_list)
# ═══════════════════════════════════════════════════════════════════════════════

def build_sequence(text: str, progress_cb=None) -> list[dict]:
    """
    For each word in `text`:
      1. Try to fetch a word-level GIF from signasl.org
      2. If not found, fall back to letter-by-letter fingerspelling images
    Returns a list of dicts: {label, kind, frames}
    """
    words   = text.strip().split()
    result  = []
    total   = sum(len(w) for w in words) + len(words)
    done    = 0

    for word in words:
        if progress_cb:
            progress_cb(done / max(total, 1), f"Looking up {word}")

        # Try word GIF first
        frames = get_word_gif_frames(word)
        if frames:
            result.append({
                "label": word.upper(),
                "kind":  "word_gif",
                "frames": [fit_image(f, CANVAS_W, CANVAS_H) for f in frames],
                "note":  "Word sign from signasl.org",
            })
            done += len(word)
            if progress_cb:
                progress_cb(done / max(total, 1), f"✓ Word sign for {word}")
            # Add space pause
            result.append({"label": "  ", "kind": "space", "frames": [], "note": ""})
            done += 1
            continue

        # Fall back to letter images
        if progress_cb:
            progress_cb(done / max(total, 1), f"Spelling {word} letter by letter…")

        for ch in word.upper():
            if not ch.isalpha():
                continue
            img = get_letter_image(ch)
            if img:
                frame = fit_image(img, CANVAS_W, CANVAS_H)
            else:
                frame = make_placeholder_frame(ch, f"ASL letter {ch}")
            result.append({
                "label":  ch,
                "kind":   "letter",
                "frames": [frame],
                "note":   f"Fingerspelling: {ch}",
            })
            done += 1
            if progress_cb:
                progress_cb(done / max(total, 1), f"Letter {ch}")

        # Space between words
        result.append({"label": "  ", "kind": "space", "frames": [], "note": ""})
        done += 1

    return result


# ═══════════════════════════════════════════════════════════════════════════════
#  Tkinter App
# ═══════════════════════════════════════════════════════════════════════════════

class ASLApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("ASL Sign Language Translator")
        self.configure(bg=BG)
        self.resizable(True, True)
        self.minsize(780, 640)

        # State
        self._sequence:   list[dict] = []
        self._idx:        int        = 0
        self._playing:    bool       = False
        self._gif_job               = None
        self._gif_frame_idx:  int   = 0
        self._photo_ref             = None  # keep reference to avoid GC
        self._build_thread          = None

        self._speed_var  = tk.DoubleVar(value=1.5)
        self._status_var = tk.StringVar(value="Enter text and press ▶ Play")

        self._build_ui()

    # ── UI construction ────────────────────────────────────────────────────────

    def _build_ui(self):
        # ── Top header ────────────────────────────────────────────────────────
        header = tk.Frame(self, bg=BG)
        header.pack(fill="x", padx=20, pady=(18, 0))

        badge = tk.Label(header, text="ASL", bg=ACCENT, fg=BG,
                         font=("Courier", 14, "bold"), padx=10, pady=4)
        badge.pack(side="left")

        tk.Label(header, text="  Sign Language Translator",
                 bg=BG, fg=TEXT, font=("Helvetica", 18, "bold")).pack(side="left")
        tk.Label(header, text="American Sign Language · Fingerspelling + Word Signs",
                 bg=BG, fg=MUTED, font=("Helvetica", 10)).pack(side="left", padx=12)

        # ── Main body ─────────────────────────────────────────────────────────
        body = tk.Frame(self, bg=BG)
        body.pack(fill="both", expand=True, padx=20, pady=14)

        body.columnconfigure(0, weight=1)
        body.columnconfigure(1, weight=2)
        body.rowconfigure(0, weight=1)

        # ── Left panel ────────────────────────────────────────────────────────
        left = tk.Frame(body, bg=SURFACE, bd=1, relief="flat",
                        highlightbackground=BORDER, highlightthickness=1)
        left.grid(row=0, column=0, sticky="nsew", padx=(0, 10))

        self._add_panel_header(left, "⌨  Input Text", ACCENT)

        # Text area
        txt_frame = tk.Frame(left, bg=SURFACE)
        txt_frame.pack(fill="both", expand=True, padx=10, pady=8)

        self._textbox = tk.Text(txt_frame, height=5, bg=CARD, fg=TEXT,
                                insertbackground=ACCENT, relief="flat",
                                font=("Helvetica", 15, "bold"), wrap="word",
                                bd=6, highlightthickness=0)
        self._textbox.pack(fill="both", expand=True)
        self._textbox.insert("1.0", "Hello World")
        self._textbox.bind("<Return>", lambda e: (self._on_play(), "break")[1])

        # Token row (scrollable)
        tk.Label(left, text="Sign sequence:", bg=SURFACE, fg=MUTED,
                 font=("Helvetica", 9)).pack(anchor="w", padx=12, pady=(4, 0))

        token_outer = tk.Frame(left, bg=SURFACE)
        token_outer.pack(fill="x", padx=10, pady=4)
        self._token_canvas = tk.Canvas(token_outer, bg=SURFACE, height=44,
                                       highlightthickness=0)
        self._token_canvas.pack(fill="x")
        self._token_frame = tk.Frame(self._token_canvas, bg=SURFACE)
        self._token_canvas.create_window((0, 0), window=self._token_frame, anchor="nw")
        self._token_frame.bind("<Configure>", lambda e: self._token_canvas.config(
            scrollregion=self._token_canvas.bbox("all")))

        # Speed slider
        spd_row = tk.Frame(left, bg=SURFACE)
        spd_row.pack(fill="x", padx=12, pady=6)
        tk.Label(spd_row, text="Speed", bg=SURFACE, fg=MUTED,
                 font=("Courier", 9)).pack(side="left")
        tk.Scale(spd_row, from_=0.5, to=4.0, resolution=0.5,
                 variable=self._speed_var, orient="horizontal",
                 bg=SURFACE, fg=TEXT, troughcolor=CARD, highlightthickness=0,
                 activebackground=ACCENT, font=("Helvetica", 8),
                 length=120, showvalue=True).pack(side="left", padx=8)
        tk.Label(spd_row, text="× (signs/sec)", bg=SURFACE, fg=MUTED,
                 font=("Courier", 9)).pack(side="left")

        # Buttons
        btn_row = tk.Frame(left, bg=SURFACE)
        btn_row.pack(fill="x", padx=10, pady=(4, 12))

        self._btn_play  = self._mk_btn(btn_row, "▶  Play",  ACCENT,  BG,  self._on_play)
        self._btn_pause = self._mk_btn(btn_row, "⏸  Pause", CARD,    TEXT, self._on_pause)
        self._btn_stop  = self._mk_btn(btn_row, "■  Stop",  ACCENT2, BG,  self._on_stop)
        for b in (self._btn_play, self._btn_pause, self._btn_stop):
            b.pack(side="left", padx=3, ipady=4, ipadx=6)

        # ── Right panel: video canvas ─────────────────────────────────────────
        right = tk.Frame(body, bg=SURFACE, bd=1, relief="flat",
                         highlightbackground=BORDER, highlightthickness=1)
        right.grid(row=0, column=1, sticky="nsew")

        self._add_panel_header(right, "🖐  Sign Display", GREEN)

        self._canvas = tk.Canvas(right, width=CANVAS_W, height=CANVAS_H,
                                 bg="#0d1117", highlightthickness=0)
        self._canvas.pack(padx=12, pady=10)
        self._draw_idle_screen()

        # progress bar
        self._progress_var = tk.DoubleVar(value=0)
        prog_frame = tk.Frame(right, bg=SURFACE)
        prog_frame.pack(fill="x", padx=12, pady=0)
        self._progress_bar = ttk.Progressbar(prog_frame, variable=self._progress_var,
                                             maximum=100, length=CANVAS_W,
                                             mode="determinate")
        self._progress_bar.pack(fill="x")

        # sign info bar
        info = tk.Frame(right, bg=CARD)
        info.pack(fill="x", padx=12, pady=8)

        self._sign_letter_lbl = tk.Label(info, text="—", bg=CARD, fg=ACCENT,
                                         font=("Courier", 36, "bold"), width=4)
        self._sign_letter_lbl.pack(side="left", padx=10)

        info_text = tk.Frame(info, bg=CARD)
        info_text.pack(side="left", fill="both", expand=True)
        self._sign_note_lbl = tk.Label(info_text, text="Enter text and press Play",
                                       bg=CARD, fg=MUTED, font=("Helvetica", 11),
                                       anchor="w", wraplength=300, justify="left")
        self._sign_note_lbl.pack(anchor="w")
        self._sign_counter_lbl = tk.Label(info_text, text="",
                                          bg=CARD, fg=MUTED, font=("Courier", 9),
                                          anchor="w")
        self._sign_counter_lbl.pack(anchor="w")

        # ── Bottom status bar ─────────────────────────────────────────────────
        status_bar = tk.Frame(self, bg=CARD)
        status_bar.pack(fill="x", padx=20, pady=(0, 12))

        self._status_dot = tk.Label(status_bar, text="●", bg=CARD, fg=MUTED,
                                    font=("Helvetica", 12))
        self._status_dot.pack(side="left", padx=(8, 4))
        tk.Label(status_bar, textvariable=self._status_var, bg=CARD, fg=MUTED,
                 font=("Helvetica", 10), anchor="w").pack(side="left", fill="x",
                                                          expand=True)
        tk.Label(status_bar, text="Cache: " + CACHE_DIR, bg=CARD, fg=BORDER,
                 font=("Courier", 8)).pack(side="right", padx=8)

    def _add_panel_header(self, parent, title, dot_color):
        hdr = tk.Frame(parent, bg=BORDER, height=1)
        hdr.pack(fill="x")
        row = tk.Frame(parent, bg=SURFACE)
        row.pack(fill="x", padx=10, pady=8)
        tk.Label(row, text="●", bg=SURFACE, fg=dot_color,
                 font=("Helvetica", 10)).pack(side="left")
        tk.Label(row, text="  " + title, bg=SURFACE, fg=MUTED,
                 font=("Courier", 9, "bold")).pack(side="left")

    def _mk_btn(self, parent, text, bg, fg, cmd):
        return tk.Button(parent, text=text, bg=bg, fg=fg, relief="flat",
                         font=("Courier", 10, "bold"), cursor="hand2",
                         activebackground=ACCENT, activeforeground=BG,
                         command=cmd, bd=0)

    # ── Idle screen ───────────────────────────────────────────────────────────

    def _draw_idle_screen(self):
        self._canvas.delete("all")
        self._canvas.create_text(CANVAS_W // 2, CANVAS_H // 2 - 20,
                                 text="🤟", font=("Helvetica", 72), fill=MUTED,
                                 anchor="center")
        self._canvas.create_text(CANVAS_W // 2, CANVAS_H // 2 + 55,
                                 text="Type text above and press ▶ Play",
                                 font=("Helvetica", 13), fill=MUTED, anchor="center")

    # ── Playback controls ─────────────────────────────────────────────────────

    def _on_play(self):
        text = self._textbox.get("1.0", "end").strip()
        if not text:
            return
        self._stop_all()
        self._set_status("⏳ Loading signs…", ACCENT)
        self._sequence = []
        self._idx      = 0

        def _load():
            seq = build_sequence(text, progress_cb=self._load_progress)
            self.after(0, lambda: self._start_playback(seq))

        self._build_thread = threading.Thread(target=_load, daemon=True)
        self._build_thread.start()

    def _load_progress(self, pct: float, msg: str):
        self.after(0, lambda: (
            self._progress_var.set(pct * 100),
            self._status_var.set(msg),
        ))

    def _start_playback(self, seq: list[dict]):
        self._sequence = seq
        self._build_token_row()
        self._idx     = 0
        self._playing = True
        self._set_status("▶ Playing", GREEN)
        self._advance()

    def _on_pause(self):
        if self._playing:
            self._playing = False
            self._cancel_gif_job()
            self._set_status("⏸ Paused", ACCENT)
        else:
            if self._sequence:
                self._playing = True
                self._set_status("▶ Playing", GREEN)
                self._advance()

    def _on_stop(self):
        self._stop_all()
        self._draw_idle_screen()
        self._sign_letter_lbl.config(text="—")
        self._sign_note_lbl.config(text="Enter text and press Play")
        self._sign_counter_lbl.config(text="")
        self._progress_var.set(0)
        self._clear_tokens()

    def _stop_all(self):
        self._playing = False
        self._cancel_gif_job()

    def _cancel_gif_job(self):
        if self._gif_job:
            self.after_cancel(self._gif_job)
            self._gif_job = None

    # ── Sequence player ───────────────────────────────────────────────────────

    def _advance(self):
        if not self._playing:
            return
        if self._idx >= len(self._sequence):
            self._playing = False
            self._set_status("✓ Done", GREEN)
            return

        item = self._sequence[self._idx]

        # Skip space items quickly
        if item["kind"] == "space":
            self._idx += 1
            delay = int(300 / self._speed_var.get())
            self._gif_job = self.after(delay, self._advance)
            return

        # Update info labels
        self._sign_letter_lbl.config(text=item["label"])
        self._sign_note_lbl.config(text=item["note"])
        self._sign_counter_lbl.config(
            text=f"{self._idx + 1} / {len(self._sequence)}  ·  "
                 f"{'Word GIF' if item['kind']=='word_gif' else 'Fingerspelling'}")
        pct = (self._idx + 1) / max(len(self._sequence), 1) * 100
        self._progress_var.set(pct)
        self._highlight_token(self._idx)

        frames = item.get("frames", [])
        if not frames:
            # blank frame
            self._canvas.delete("all")
            self._idx += 1
            self._gif_job = self.after(250, self._advance)
            return

        if item["kind"] == "word_gif" and len(frames) > 1:
            # animated GIF: play all frames then move on
            self._play_gif_frames(frames, 0, item)
        else:
            # static image: show and wait
            self._show_frame(frames[0])
            hold_ms = int(1000 / self._speed_var.get())
            self._idx += 1
            self._gif_job = self.after(hold_ms, self._advance)

    def _play_gif_frames(self, frames, frame_idx, item):
        if not self._playing:
            return
        self._show_frame(frames[frame_idx])
        next_frame = frame_idx + 1
        if next_frame < len(frames):
            # ~60ms per frame default gif frame delay
            delay = max(40, int(60 / self._speed_var.get()))
            self._gif_job = self.after(delay, lambda: self._play_gif_frames(
                frames, next_frame, item))
        else:
            # GIF finished → pause 300ms then next item
            self._idx += 1
            self._gif_job = self.after(300, self._advance)

    def _show_frame(self, img: Image.Image):
        photo = ImageTk.PhotoImage(img)
        self._photo_ref = photo  # prevent GC
        self._canvas.delete("all")
        self._canvas.create_image(0, 0, anchor="nw", image=photo)

    # ── Token row (letter/word chips) ─────────────────────────────────────────

    def _build_token_row(self):
        self._clear_tokens()
        self._token_labels = []
        for i, item in enumerate(self._sequence):
            lbl = item["label"].strip()
            if not lbl:
                lbl = "·"
            is_word = item["kind"] == "word_gif"
            bg_col  = "#1f3a5f" if is_word else CARD
            txt_col = ACCENT if is_word else TEXT
            chip = tk.Label(self._token_frame, text=lbl,
                            bg=bg_col, fg=txt_col,
                            font=("Courier", 10, "bold"),
                            padx=6, pady=2, relief="flat",
                            cursor="hand2")
            chip.pack(side="left", padx=2, pady=4)
            idx = i  # capture
            chip.bind("<Button-1>", lambda e, n=idx: self._jump_to(n))
            self._token_labels.append(chip)

    def _clear_tokens(self):
        for w in self._token_frame.winfo_children():
            w.destroy()
        self._token_labels = []

    def _highlight_token(self, idx):
        for i, chip in enumerate(self._token_labels):
            if i == idx:
                chip.config(bg=ACCENT, fg=BG)
            elif i < idx:
                chip.config(bg="#1c2d42" if self._sequence[i]["kind"] == "word_gif"
                            else "#1e2a1e",
                            fg=MUTED)
            else:
                is_word = self._sequence[i]["kind"] == "word_gif"
                chip.config(bg="#1f3a5f" if is_word else CARD,
                            fg=ACCENT if is_word else TEXT)

    def _jump_to(self, idx):
        self._cancel_gif_job()
        self._idx = idx
        if not self._playing:
            self._playing = True
            self._set_status("▶ Playing", GREEN)
        self._advance()

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _set_status(self, msg: str, dot_color: str = MUTED):
        self._status_var.set(msg)
        self._status_dot.config(fg=dot_color)


# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    app = ASLApp()
    app.mainloop()

invalid command name "140320542228800_next_token"
    while executing
"140320542228800_next_token"
    ("after" script)


In [1]:
import tkinter as tk
from tkinter import ttk, font
import threading
import os
import time
from PIL import Image, ImageTk, ImageSequence
import speech_recognition as sr

# ─────────────────────────────────────────────
#  PATHS  (edit these to match your system)
# ─────────────────────────────────────────────
LETTER_FOLDER = "/home/maries/Development/Project_OWSLT/assets/ASl letters/"
GIF_FOLDER    = "/home/maries/Development/Project_OWSLT/assets/ASL gif/"

# GIFs that exist in your gif folder (lowercase, no extension)
GIF_WORDS = {
    "HELLO", "THANKS", "WATER", "GO", "UNDERSTAND", "EAT", "FRIEND",
    "HELP", "GOODBYE", "MORNING", "GOODNIGHT", "I", "PLEASE", "SORRY",
    "YES", "YOU", "FOOD", "SAD", "GOODEVENING", "DRINK", "NO",
}

# Map spoken/typed variants → canonical gif name
WORD_ALIASES = {
    "good evening": "goodevening",
    "good night":   "goodnight",
    "good morning": "morning",
    "good bye":     "goodbye",
    "thank you":    "thanks",
    "thank":        "thanks",
}

# ─────────────────────────────────────────────
#  COLOURS / FONTS
# ─────────────────────────────────────────────
BG         = "#0d1117"
PANEL      = "#161b22"
ACCENT     = "#58a6ff"
ACCENT2    = "#3fb950"
TEXT       = "#e6edf3"
SUBTEXT    = "#8b949e"
BORDER     = "#30363d"
BTN_BG     = "#21262d"
BTN_HOVER  = "#30363d"
DANGER     = "#f85149"

DISPLAY_W, DISPLAY_H = 380, 340   # main sign display area

# ─────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────
def find_gif(word: str):
    """Return path to gif if found, else None."""
    w = word.upper().strip()
    w = WORD_ALIASES.get(w, w)
    if w in GIF_WORDS:
        for ext in (".gif", ".GIF"):
            p = os.path.join(GIF_FOLDER, w + ext)
            if os.path.exists(p):
                return p
    return None

def find_letter_image(letter: str):
    """Return path to letter image for a single character."""
    l = letter.upper()
    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"):
        p = os.path.join(LETTER_FOLDER, l + ext)
        if os.path.exists(p):
            return p
    # Try lowercase filename too
    l2 = letter.lower()
    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"):
        p = os.path.join(LETTER_FOLDER, l2 + ext)
        if os.path.exists(p):
            return p
    return None

def tokenise(text: str):
    """
    Split text into a list of tokens.
    Multi-word gif matches are tried first (up to 2 words).
    Returns list of ('gif', path, label) or ('letter', path, char) or ('space',) tuples.
    """
    words = text.upper().strip().split()
    tokens = []
    i = 0
    while i < len(words):
        # try 2-word match first
        matched = False
        if i + 1 < len(words):
            two = words[i] + " " + words[i + 1]
            alias = WORD_ALIASES.get(two, two)
            gp = find_gif(alias)
            if gp:
                tokens.append(("gif", gp, two))
                i += 2
                matched = True
        if not matched:
            w = words[i]
            alias = WORD_ALIASES.get(w, w)
            gp = find_gif(alias)
            if gp:
                tokens.append(("gif", gp, w))
            else:
                # spell it out letter by letter
                for ch in w:
                    if ch.isalpha():
                        lp = find_letter_image(ch)
                        tokens.append(("letter", lp, ch))
            i += 1
        # add space between words (except at end)
        if i < len(words):
            tokens.append(("space",))
    return tokens


# ─────────────────────────────────────────────
#  ANIMATED GIF PLAYER  (frame-by-frame)
# ─────────────────────────────────────────────
class GifPlayer:
    def __init__(self, label: tk.Label, path: str, w: int, h: int, on_done):
        self.label   = label
        self.on_done = on_done
        self.frames  = []
        self.delays  = []
        self.running = False
        self._load(path, w, h)

    def _load(self, path, w, h):
        try:
            img = Image.open(path)
            for frame in ImageSequence.Iterator(img):
                f = frame.convert("RGBA").resize((w, h), Image.LANCZOS)
                self.frames.append(ImageTk.PhotoImage(f))
                try:
                    d = frame.info.get("duration", 80)
                except Exception:
                    d = 80
                self.delays.append(max(int(d), 40))
        except Exception as e:
            print(f"[GifPlayer] could not load {path}: {e}")

    def play(self):
        if not self.frames:
            self.on_done()
            return
        self.running = True
        self._idx = 0
        self._advance()

    def stop(self):
        self.running = False

    def _advance(self):
        if not self.running:
            return
        if self._idx < len(self.frames):
            self.label.configure(image=self.frames[self._idx])
            self.label._img = self.frames[self._idx]   # keep ref
            delay = self.delays[self._idx]
            self._idx += 1
            self.label.after(delay, self._advance)
        else:
            self.running = False
            self.on_done()


# ─────────────────────────────────────────────
#  MAIN APP
# ─────────────────────────────────────────────
class ASLApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("ASL Sign Generator")
        self.configure(bg=BG)
        self.resizable(False, False)

        self._tokens      = []
        self._tok_idx     = 0
        self._playing     = False
        self._gif_player  = None
        self._static_img  = None   # keep PhotoImage ref alive
        self._letter_imgs = []     # row of letter thumbnails

        self._build_ui()
        self.update_idletasks()
        self._center()

    # ── layout ──────────────────────────────
    def _build_ui(self):
        # ── title bar ──
        top = tk.Frame(self, bg=BG, pady=14)
        top.pack(fill="x")
        tk.Label(top, text=" ASL Sign Generator",
                 font=("Courier New", 18, "bold"),
                 fg=ACCENT, bg=BG).pack()
        tk.Label(top, text="Text or Speech → American Sign Language",
                 font=("Courier New", 10),
                 fg=SUBTEXT, bg=BG).pack()

        # ── divider ──
        tk.Frame(self, bg=BORDER, height=1).pack(fill="x")

        # ── input area ──
        inp_frame = tk.Frame(self, bg=PANEL, pady=16, padx=20)
        inp_frame.pack(fill="x")

        tk.Label(inp_frame, text="Type your text:", font=("Courier New", 10),
                 fg=SUBTEXT, bg=PANEL).pack(anchor="w")

        entry_row = tk.Frame(inp_frame, bg=PANEL)
        entry_row.pack(fill="x", pady=(6, 0))

        self.entry = tk.Entry(entry_row,
                              font=("Courier New", 13),
                              bg=BTN_BG, fg=TEXT,
                              insertbackground=ACCENT,
                              relief="flat",
                              bd=0,
                              highlightthickness=1,
                              highlightbackground=BORDER,
                              highlightcolor=ACCENT)
        self.entry.pack(side="left", fill="x", expand=True, ipady=8, padx=(0, 10))
        self.entry.bind("<Return>", lambda e: self._on_convert())

        self._btn_convert = self._make_btn(entry_row, "▶  Convert",
                                           ACCENT, self._on_convert)
        self._btn_convert.pack(side="left")

        # speech button
        self._btn_speech = self._make_btn(inp_frame, "🎤  Speak",
                                          ACCENT2, self._on_speech, full=True)
        self._btn_speech.pack(fill="x", pady=(10, 0))

        # ── divider ──
        tk.Frame(self, bg=BORDER, height=1).pack(fill="x")

        # ── display area ──
        disp_outer = tk.Frame(self, bg=BG, pady=20, padx=20)
        disp_outer.pack()

        # progress / word label
        self.lbl_word = tk.Label(disp_outer,
                                 text="Enter text and press Convert",
                                 font=("Courier New", 12, "bold"),
                                 fg=ACCENT, bg=BG)
        self.lbl_word.pack()

        # main sign canvas
        self.sign_canvas = tk.Frame(disp_outer, bg=PANEL,
                                    highlightthickness=2,
                                    highlightbackground=BORDER,
                                    width=DISPLAY_W, height=DISPLAY_H)
        self.sign_canvas.pack(pady=(10, 0))
        self.sign_canvas.pack_propagate(False)

        self.sign_lbl = tk.Label(self.sign_canvas, bg=PANEL,
                                 text="", font=("Segoe UI Emoji", 64))
        self.sign_lbl.pack(expand=True)

        # letter strip (fingerspelling)
        self.letter_strip = tk.Frame(disp_outer, bg=BG)
        self.letter_strip.pack(pady=(14, 0))

        # status
        self.lbl_status = tk.Label(disp_outer, text="",
                                   font=("Courier New", 9),
                                   fg=SUBTEXT, bg=BG)
        self.lbl_status.pack(pady=(8, 0))

        # token progress bar row
        self.progress_frame = tk.Frame(disp_outer, bg=BG)
        self.progress_frame.pack(pady=(6, 0))

    def _make_btn(self, parent, text, color, cmd, full=False):
        btn = tk.Label(parent, text=text,
                       font=("Courier New", 11, "bold"),
                       fg=color, bg=BTN_BG,
                       cursor="hand2",
                       padx=16, pady=8,
                       relief="flat",
                       bd=0,
                       highlightthickness=1,
                       highlightbackground=color)
        btn.bind("<Button-1>", lambda e: cmd())
        btn.bind("<Enter>",    lambda e: btn.configure(bg=BTN_HOVER))
        btn.bind("<Leave>",    lambda e: btn.configure(bg=BTN_BG))
        return btn

    def _center(self):
        self.update_idletasks()
        sw = self.winfo_screenwidth()
        sh = self.winfo_screenheight()
        w  = self.winfo_width()
        h  = self.winfo_height()
        self.geometry(f"+{(sw-w)//2}+{(sh-h)//2}")

    # ── actions ─────────────────────────────
    def _on_convert(self):
        text = self.entry.get().strip()
        if not text:
            return
        self._start_playback(text)

    def _on_speech(self):
        self._btn_speech.configure(text="  Listening…", fg=DANGER)
        self.update()
        threading.Thread(target=self._listen, daemon=True).start()

    def _listen(self):
        r = sr.Recognizer()
        try:
            with sr.Microphone() as source:
                r.adjust_for_ambient_noise(source, duration=0.5)
                audio = r.listen(source, timeout=6, phrase_time_limit=8)
            text = r.recognize_google(audio)
            self.after(0, lambda: self.entry.delete(0, "end"))
            self.after(0, lambda: self.entry.insert(0, text))
            self.after(0, lambda: self._start_playback(text))
        except sr.WaitTimeoutError:
            self.after(0, lambda: self._set_status(" No speech detected.", DANGER))
        except sr.UnknownValueError:
            self.after(0, lambda: self._set_status(" Could not understand audio.", DANGER))
        except Exception as ex:
            self.after(0, lambda: self._set_status(f"Mic error: {ex}", DANGER))
        finally:
            self.after(0, lambda: self._btn_speech.configure(
                text=" Speak", fg=ACCENT2))

    # ── playback engine ─────────────────────
    def _start_playback(self, text: str):
        if self._gif_player:
            self._gif_player.stop()
        self._playing = True
        self._tokens  = tokenise(text)
        self._tok_idx = 0
        self._build_progress_dots()
        self._next_token()

    def _build_progress_dots(self):
        for w in self.progress_frame.winfo_children():
            w.destroy()
        self._dots = []
        for t in self._tokens:
            if t[0] == "space":
                continue
            dot = tk.Label(self.progress_frame, text="●",
                           font=("Courier New", 7),
                           fg=BORDER, bg=BG)
            dot.pack(side="left", padx=1)
            self._dots.append(dot)
        self._dot_idx = 0

    def _light_next_dot(self):
        if self._dot_idx < len(self._dots):
            self._dots[self._dot_idx].configure(fg=ACCENT)
            self._dot_idx += 1

    def _next_token(self):
        # clear letter strip
        for w in self.letter_strip.winfo_children():
            w.destroy()

        if self._tok_idx >= len(self._tokens):
            self._playing = False
            self._set_status(" Done!", ACCENT2)
            self.lbl_word.configure(text="Finished")
            return

        tok = self._tokens[self._tok_idx]
        self._tok_idx += 1

        if tok[0] == "space":
            self.after(200, self._next_token)
            return

        self._light_next_dot()

        if tok[0] == "gif":
            _, path, label = tok
            self.lbl_word.configure(text=f" {label.upper()}")
            self._set_status(f"Playing GIF: {label}")
            self._play_gif(path)

        elif tok[0] == "letter":
            _, path, ch = tok
            self.lbl_word.configure(text=f"  Fingerspelling: {ch.upper()}")
            self._set_status(f"Letter: {ch.upper()}")
            self._show_letter(path, ch)

    def _play_gif(self, path: str):
        self.sign_lbl.configure(text="", image="")
        self._gif_player = GifPlayer(
            self.sign_lbl, path, DISPLAY_W - 10, DISPLAY_H - 10,
            on_done=lambda: self.after(300, self._next_token)
        )
        self._gif_player.play()

    def _show_letter(self, path, ch: str):
        if path and os.path.exists(path):
            try:
                img = Image.open(path).convert("RGBA")
                img.thumbnail((DISPLAY_W - 20, DISPLAY_H - 20), Image.LANCZOS)
                self._static_img = ImageTk.PhotoImage(img)
                self.sign_lbl.configure(image=self._static_img, text="")
            except Exception as e:
                self._show_fallback(ch)
                print(f"[Letter] {e}")
        else:
            self._show_fallback(ch)
        # show letter card in strip
        self._add_letter_strip_card(path, ch)
        self.after(900, self._next_token)

    def _show_fallback(self, ch: str):
        self.sign_lbl.configure(
            image="",
            text=ch.upper(),
            font=("Courier New", 96, "bold"),
            fg=ACCENT
        )

    def _add_letter_strip_card(self, path, ch: str):
        card = tk.Frame(self.letter_strip, bg=PANEL,
                        highlightthickness=1, highlightbackground=BORDER)
        card.pack(side="left", padx=3)

        if path and os.path.exists(path):
            try:
                img = Image.open(path).convert("RGBA")
                img.thumbnail((48, 48), Image.LANCZOS)
                ph = ImageTk.PhotoImage(img)
                lbl = tk.Label(card, image=ph, bg=PANEL)
                lbl.image = ph
                lbl.pack(padx=4, pady=4)
            except Exception:
                tk.Label(card, text=ch.upper(),
                         font=("Courier New", 20, "bold"),
                         fg=ACCENT, bg=PANEL,
                         padx=6, pady=6).pack()
        else:
            tk.Label(card, text=ch.upper(),
                     font=("Courier New", 20, "bold"),
                     fg=ACCENT, bg=PANEL,
                     padx=6, pady=6).pack()

        tk.Label(card, text=ch.upper(),
                 font=("Courier New", 8),
                 fg=SUBTEXT, bg=PANEL).pack(pady=(0, 4))
        self._letter_imgs.append(card)   # keep ref

    def _set_status(self, msg: str, color: str = SUBTEXT):
        self.lbl_status.configure(text=msg, fg=color)


# ─────────────────────────────────────────────
#  ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    app = ASLApp()
    app.mainloop()

In [1]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║         ASL BIDIRECTIONAL CONVERSATION SYSTEM                        ║
║                                                                      ║
║  MODE A  ─  Sign → Text / TTS                                        ║
║    Deaf/HoH user signs via camera; ResNet (letter) + LSTM (word)     ║
║    detects signs, fills sentence buffer, then confirms to speak/show ║
║                                                                      ║
║  MODE B  ─  Text → Sign                                              ║
║    Hearing user's reply typed/spoken; system plays ASL GIF / finger- ║
║    spelling so the deaf user can read the response                   ║
║                                                                      ║
║  FLOW  ─  Alternating conversation turns with confirm dialogs:       ║
║    [Sign→Text] → confirm → speak/show text → switch alert            ║
║    [Text→Sign] → confirm → play ASL signs → switch alert → repeat    ║
║                                                                      ║
║  Requirements:                                                       ║
║    pip install customtkinter opencv-python mediapipe tensorflow      ║
║               pyttsx3 pillow requests speechrecognition              ║
╚══════════════════════════════════════════════════════════════════════╝
"""

import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp
from collections import deque
import pyttsx3
import time
import threading
import tkinter as tk
from tkinter import ttk
import customtkinter as ctk
from PIL import Image, ImageTk, ImageDraw, ImageFont, ImageSequence
import os
import re
import requests
from io import BytesIO
import speech_recognition as sr

# ──────────────────────────────────────────────────────────────────────────────
#  PATHS  ── edit to match your system
# ──────────────────────────────────────────────────────────────────────────────

MODEL_LETTER = '/home/maries/Development/Project_OWSLT/models/letter_modelV2.keras'
MODEL_WORD   = '/home/maries/Development/Project_OWSLT/models/word_modelV2.keras'
LETTER_FOLDER = '/home/maries/Development/Project_OWSLT/assets/ASl letters/'
GIF_FOLDER    = '/home/maries/Development/Project_OWSLT/assets/ASL gif/'

CACHE_DIR = os.path.join(os.path.expanduser("~"), ".asl_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
#  PALETTE
# ──────────────────────────────────────────────────────────────────────────────

BG       = "#0d0d0f"
PANEL    = "#16161a"
CARD     = "#1c1c22"
BORDER   = "#2a2a35"
ACCENT   = "#00e5ff"    # sign→text  (cyan)
ACCENT2  = "#ff3d71"    # text→sign  (rose)
GREEN    = "#00ff9d"
MUTED    = "#6b6b80"
TEXT     = "#e8e8f0"
WHITE    = "#ffffff"
BTN_H    = "#1a1a22"
WARN_BG  = "#1a1200"
WARN_FG  = "#ffcc00"

# ──────────────────────────────────────────────────────────────────────────────
#  ASL HAND SKELETON HELPERS
# ──────────────────────────────────────────────────────────────────────────────

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),
]

FINGER_COLORS = {
    (0,1):(0,255,255),(1,2):(0,255,255),(2,3):(0,255,255),(3,4):(0,255,255),
    (0,5):(255,80,80),(5,6):(255,80,80),(6,7):(255,80,80),(7,8):(255,80,80),
    (0,9):(80,255,80),(9,10):(80,255,80),(10,11):(80,255,80),(11,12):(80,255,80),
    (0,13):(80,80,255),(13,14):(80,80,255),(14,15):(80,80,255),(15,16):(80,80,255),
    (0,17):(255,80,255),(17,18):(255,80,255),(18,19):(255,80,255),(19,20):(255,80,255),
    (5,9):(220,220,220),(9,13):(220,220,220),(13,17):(220,220,220),
}

def normalize_landmarks(landmarks, img_w, img_h):
    points = np.array([[lm.x * img_w, lm.y * img_h] for lm in landmarks])
    mn = points.min(axis=0); mx = points.max(axis=0)
    cx = (mn[0]+mx[0])/2; cy = (mn[1]+mx[1])/2
    dim = max(mx[0]-mn[0], mx[1]-mn[1])
    scale = (min(img_w,img_h)*0.8)/dim if dim > 0 else 1
    return [(int((p[0]-cx)*scale+img_w/2), int((p[1]-cy)*scale+img_h/2)) for p in points]

def draw_hand_skeleton(frame, pts):
    for c in HAND_CONNECTIONS:
        cv2.line(frame, pts[c[0]], pts[c[1]], FINGER_COLORS[c], 3)
    for i, p in enumerate(pts):
        color = (0,0,255) if i in [4,8,12,16,20] else (255,0,255)
        cv2.circle(frame, p, 3, color, -1)
        cv2.circle(frame, p, 3, (255,255,255), 1)
    return frame

mp_holistic   = mp.solutions.holistic
mp_hands_mod  = mp.solutions.hands

def extract_keypoints(results):
    pose = (np.array([[r.x,r.y,r.z,r.visibility] for r in results.pose_landmarks.landmark]).flatten()
            if results.pose_landmarks else np.zeros(132))
    lh   = (np.array([[r.x,r.y,r.z] for r in results.left_hand_landmarks.landmark]).flatten()
            if results.left_hand_landmarks else np.zeros(63))
    rh   = (np.array([[r.x,r.y,r.z] for r in results.right_hand_landmarks.landmark]).flatten()
            if results.right_hand_landmarks else np.zeros(63))
    return np.concatenate([pose, lh, rh])

# ──────────────────────────────────────────────────────────────────────────────
#  TEXT → SIGN  HELPERS  (from ASL Sign Generator)
# ──────────────────────────────────────────────────────────────────────────────

GIF_WORDS = {
    "HELLO","THANKS","WATER","GO","UNDERSTAND","EAT","FRIEND",
    "HELP","GOODBYE","MORNING","GOODNIGHT","I","PLEASE","SORRY",
    "YES","YOU","FOOD","SAD","GOODEVENING","DRINK","NO",
}

WORD_ALIASES = {
    "good evening":"goodevening", "good night":"goodnight",
    "good morning":"morning",     "good bye":"goodbye",
    "thank you":"thanks",         "thank":"thanks",
}

SIGNASL_SEARCH = "https://www.signasl.org/sign/{word}"

CANVAS_W, CANVAS_H = 460, 340

def cache_path(key, ext):
    safe = re.sub(r'[^a-zA-Z0-9_\-]', '_', key)
    return os.path.join(CACHE_DIR, f"{safe}.{ext}")

def fetch_bytes(url, timeout=8):
    try:
        r = requests.get(url, timeout=timeout,
                         headers={"User-Agent": "Mozilla/5.0 (ASL-App/2.0)"})
        if r.status_code == 200:
            return r.content
    except Exception:
        pass
    return None

def find_gif_local(word):
    w = word.upper().strip()
    w = WORD_ALIASES.get(w, w)
    if w in GIF_WORDS:
        for ext in (".gif", ".GIF"):
            p = os.path.join(GIF_FOLDER, w + ext)
            if os.path.exists(p):
                return p
    return None

def find_letter_image_local(letter):
    l = letter.upper()
    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"):
        p = os.path.join(LETTER_FOLDER, l + ext)
        if os.path.exists(p):
            return p
    return None

def get_word_gif_url_online(word):
    url  = SIGNASL_SEARCH.format(word=word.lower().strip())
    data = fetch_bytes(url, timeout=10)
    if not data:
        return None
    html = data.decode("utf-8", errors="ignore")
    for pat in [
        r'<source[^>]+src=["\']([\w:/\.\-]+\.gif)["\']',
        r'<img[^>]+src=["\']([\w:/\.\-]+\.gif)["\']',
        r'(https?://[^\s"\'<>]+\.gif)',
    ]:
        m = re.search(pat, html, re.IGNORECASE)
        if m:
            return m.group(1)
    return None

def get_letter_image_online(letter):
    letter = letter.upper()
    cp = cache_path(f"letter_{letter}", "png")
    if os.path.exists(cp):
        try:
            return Image.open(cp).convert("RGBA")
        except Exception:
            pass
    urls = [
        f"https://raw.githubusercontent.com/Temitope-py/sign-language-detector/main/data/test/{letter}/{letter}1.jpg",
        f"https://raw.githubusercontent.com/Temitope-py/sign-language-detector/main/data/test/{letter.lower()}/{letter.lower()}1.jpg",
        f"https://upload.wikimedia.org/wikipedia/commons/thumb/9/9a/Sign_language_{letter.lower()}.svg/240px-Sign_language_{letter.lower()}.svg.png",
    ]
    for url in urls:
        data = fetch_bytes(url)
        if data:
            try:
                img = Image.open(BytesIO(data)).convert("RGBA")
                img.save(cp)
                return img
            except Exception:
                continue
    return None

def fit_image(img, w, h):
    img.thumbnail((w, h), Image.LANCZOS)
    canvas = Image.new("RGBA", (w, h), (13, 13, 15, 255))
    x = (w - img.width)//2; y = (h - img.height)//2
    canvas.paste(img, (x, y), img if img.mode == "RGBA" else None)
    return canvas

def make_placeholder(text, sub=""):
    img  = Image.new("RGBA", (CANVAS_W, CANVAS_H), (13,13,15,255))
    draw = ImageDraw.Draw(img)
    try:
        fnt_big = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 110)
        fnt_sub = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 20)
    except Exception:
        fnt_big = ImageFont.load_default()
        fnt_sub = fnt_big
    bb  = draw.textbbox((0,0), text, font=fnt_big)
    tw, th = bb[2]-bb[0], bb[3]-bb[1]
    draw.text(((CANVAS_W-tw)//2, (CANVAS_H-th)//2 - 20), text, fill=(0,229,255,220), font=fnt_big)
    if sub:
        bb2 = draw.textbbox((0,0), sub, font=fnt_sub)
        draw.text(((CANVAS_W-(bb2[2]-bb2[0]))//2, CANVAS_H-50), sub, fill=(107,107,128,200), font=fnt_sub)
    return img

def build_sign_sequence(text):
    """Returns list of dicts with kind/label/frames for playback."""
    words  = text.strip().upper().split()
    result = []
    for word in words:
        # 1) Try local GIF
        lp = find_gif_local(word)
        if lp:
            try:
                img    = Image.open(lp)
                frames = [fit_image(f.copy().convert("RGBA"), CANVAS_W, CANVAS_H)
                          for f in ImageSequence.Iterator(img)]
                if frames:
                    result.append({"kind":"gif","label":word,"frames":frames,"local":True})
                    result.append({"kind":"space","label":"","frames":[]})
                    continue
            except Exception:
                pass

        # 2) Try online GIF
        gif_url = get_word_gif_url_online(word)
        if gif_url:
            cp = cache_path(f"word_{word.lower()}", "gif")
            if not os.path.exists(cp):
                data = fetch_bytes(gif_url, timeout=15)
                if data:
                    with open(cp, "wb") as f:
                        f.write(data)
            if os.path.exists(cp):
                try:
                    img    = Image.open(cp)
                    frames = [fit_image(f.copy().convert("RGBA"), CANVAS_W, CANVAS_H)
                              for f in ImageSequence.Iterator(img)]
                    if frames:
                        result.append({"kind":"gif","label":word,"frames":frames,"local":False})
                        result.append({"kind":"space","label":"","frames":[]})
                        continue
                except Exception:
                    pass

        # 3) Fall back to letter-by-letter
        for ch in word:
            if not ch.isalpha():
                continue
            img = None
            lp2 = find_letter_image_local(ch)
            if lp2:
                try:
                    img = fit_image(Image.open(lp2).convert("RGBA"), CANVAS_W, CANVAS_H)
                except Exception:
                    pass
            if img is None:
                img_pil = get_letter_image_online(ch)
                img = fit_image(img_pil, CANVAS_W, CANVAS_H) if img_pil else make_placeholder(ch, f"ASL {ch}")
            result.append({"kind":"letter","label":ch,"frames":[img]})
        result.append({"kind":"space","label":"","frames":[]})
    return result


# ──────────────────────────────────────────────────────────────────────────────
#  ANIMATED GIF PLAYER  (helper for text→sign)
# ──────────────────────────────────────────────────────────────────────────────

class GifPlayer:
    def __init__(self, widget, frames, speed_var, on_done):
        self.widget    = widget
        self.frames    = frames
        self.speed_var = speed_var
        self.on_done   = on_done
        self._idx      = 0
        self._running  = False
        self._ph_cache = [ImageTk.PhotoImage(f) for f in frames]

    def play(self):
        if not self._ph_cache:
            self.on_done()
            return
        self._running = True
        self._idx = 0
        self._advance()

    def stop(self):
        self._running = False

    def _advance(self):
        if not self._running:
            return
        if self._idx < len(self._ph_cache):
            ph = self._ph_cache[self._idx]
            self.widget.configure(image=ph)
            self.widget._img = ph
            delay = max(40, int(60 / self.speed_var.get()))
            self._idx += 1
            self.widget.after(delay, self._advance)
        else:
            self._running = False
            self.on_done()


# ──────────────────────────────────────────────────────────────────────────────
#  CUSTOM CONFIRM DIALOG  (dark themed, non-blocking via after())
# ──────────────────────────────────────────────────────────────────────────────

class ConfirmDialog(ctk.CTkToplevel):
    """
    Shows a styled modal dialog.
    on_result(choice) is called with "confirm", "tts", or "cancel".
    """
    def __init__(self, parent, title, message, buttons, on_result, accent=ACCENT):
        super().__init__(parent)
        self.on_result = on_result
        self.title(title)
        self.configure(fg_color=PANEL)
        self.resizable(False, False)
        self.grab_set()
        self.attributes("-topmost", True)

        # ── Icon + title ──
        hdr = ctk.CTkFrame(self, fg_color=BG, corner_radius=0)
        hdr.pack(fill="x")
        ctk.CTkLabel(hdr, text=f"◈  {title}",
                     font=("Courier New", 15, "bold"), text_color=accent).pack(
            padx=20, pady=14, anchor="w")

        # ── Message ──
        ctk.CTkLabel(self, text=message,
                     font=("Courier New", 12), text_color=TEXT,
                     wraplength=380, justify="left").pack(padx=24, pady=(8, 20))

        # ── Buttons ──
        btn_row = ctk.CTkFrame(self, fg_color="transparent")
        btn_row.pack(padx=20, pady=(0, 20))

        COLORS = {
            "confirm": (accent, "#0d0d0f"),
            "tts":     (GREEN,  "#0d0d0f"),
            "switch":  (WARN_FG, "#0d0d0f"),
            "cancel":  (BTN_H,  MUTED),
        }

        for key, label in buttons:
            fg, tc = COLORS.get(key, (BTN_H, TEXT))
            ctk.CTkButton(
                btn_row, text=label, width=140, height=40,
                fg_color=fg, hover_color=fg, text_color=tc,
                font=("Courier New", 12, "bold"), corner_radius=8,
                command=lambda k=key: self._respond(k)
            ).pack(side="left", padx=6)

        self._center(parent)

    def _center(self, parent):
        self.update_idletasks()
        pw = parent.winfo_width();  ph = parent.winfo_height()
        px = parent.winfo_rootx(); py = parent.winfo_rooty()
        w  = self.winfo_reqwidth(); h  = self.winfo_reqheight()
        self.geometry(f"+{px + (pw-w)//2}+{py + (ph-h)//2}")

    def _respond(self, choice):
        self.grab_release()
        self.destroy()
        self.on_result(choice)


# ──────────────────────────────────────────────────────────────────────────────
#  MAIN APPLICATION
# ──────────────────────────────────────────────────────────────────────────────

class ASLBidirectionalApp:
    """
    Two-panel application:
      LEFT  → Sign-to-Text  (camera + ML models)  [MODE A]
      RIGHT → Text-to-Sign  (GIF / fingerspelling) [MODE B]

    Conversation flow (alternates):
      1. Deaf user signs message into buffer  (MODE A active)
      2. Buffer fills → user clicks "Send Message"
         → ConfirmDialog: [Send as Text] [Speak Aloud] [Cancel]
      3. Message displayed in chat log
         → ConfirmDialog: "Switch to Text→Sign so hearing person can reply?"
           [Switch] [Cancel]
      4. Hearing person types / speaks reply  (MODE B active)
      5. User clicks "Send Reply"
         → ConfirmDialog: [Play Signs] [Cancel]
      6. Signs play out
         → ConfirmDialog: "Switch back to Sign→Text so deaf person can reply?"
           [Switch] [Cancel]
      7. Repeat
    """

    LETTER_LABELS = ['a','b','c','d','e','f','g','h','i','j','k','l','m',
                     'n','o','p','q','r','s','t','u','v','w','x','y','z',
                     'del','ok','space']
    WORD_LABELS   = ['hello','help','meet','nice','nothing','please','thanks']

    DEBOUNCE_SEC  = 1.5
    WORD_STAB_FRM = 8
    CONF_THRESH   = 0.70

    def __init__(self, root: ctk.CTk):
        self.root    = root
        self.root.title("ASL Bidirectional Conversation System")
        self.root.geometry("1400x860")
        self.root.resizable(False, False)

        # ── conversation mode: "sign_to_text" | "text_to_sign"
        self.conv_mode = "sign_to_text"

        # ── sign→text state ──
        self.sign_buffer        = ""
        self.running            = True
        self.sign_mode          = "letter"  
        self.pending_letter     = None
        self.pending_since      = None
        self.word_sequence      = deque(maxlen=30)
        self.last_word_pred     = None
        self.word_stab_count    = 0
        self.last_confirmed_word = None
        self.cur_prediction     = ""
        self.cur_confidence     = 0.0
        self.skeleton_frame     = np.zeros((224,224,3), dtype=np.uint8)

        # ── text→sign state ──
        self.sign_sequence      = []
        self.sign_seq_idx       = 0
        self.sign_playing       = False
        self.gif_player_obj     = None
        self.sign_gif_job       = None
        self._static_img_ref    = None
        self._speed_var         = tk.DoubleVar(value=1.5)

        # ── shared ──
        self.chat_log           = []   # list of {"role": "deaf"|"hearing", "text": str}

        # ── TTS ──
        self.tts_engine = pyttsx3.init()
        self.tts_engine.setProperty('rate', 150)

        # ── Models ──
        self.letter_model  = None
        self.word_model    = None
        self.holistic      = None
        self.hand_detector = None

        self._build_ui()
        self._load_models_async()

        self.cap = cv2.VideoCapture(0)
        self._camera_loop()

   
    #  UI 

    def _build_ui(self):
        self.root.configure(fg_color=BG)

        # ── top bar ──────────────────────────────────────────────────────────
        topbar = ctk.CTkFrame(self.root, fg_color=PANEL, height=56, corner_radius=0)
        topbar.pack(fill="x", side="top")
        topbar.pack_propagate(False)

        ctk.CTkLabel(topbar, text="◈  ASL BIDIRECTIONAL CONVERSATION",
                     font=("Courier New", 16, "bold"), text_color=ACCENT).pack(
            side="left", padx=20, pady=16)

        self.mode_badge = ctk.CTkLabel(topbar,
            text="● SIGN → TEXT  (Deaf Turn)",
            font=("Courier New", 12, "bold"), text_color=ACCENT)
        self.mode_badge.pack(side="right", padx=24)

        self.status_dot = ctk.CTkLabel(topbar, text="LOADING",
                                       font=("Courier New", 11), text_color="#ffaa00")
        self.status_dot.pack(side="right", padx=16)

        # ── main body ─────────────────────────────────────────────────────────
        body = ctk.CTkFrame(self.root, fg_color=BG)
        body.pack(fill="both", expand=True)

        # ── COLUMN A: Sign→Text ──────────────────────────────────────────────
        self.col_a = ctk.CTkFrame(body, fg_color=PANEL, corner_radius=14,
                                   border_color=ACCENT, border_width=2)
        self.col_a.pack(side="left", fill="both", expand=True,
                        padx=(12,6), pady=12)

        self._build_sign_to_text_panel(self.col_a)

        # ── COLUMN M: Chat log (middle) ──────────────────────────────────────
        col_m = ctk.CTkFrame(body, fg_color=CARD, corner_radius=14, width=220)
        col_m.pack(side="left", fill="both", padx=6, pady=12)
        col_m.pack_propagate(False)
        self._build_chat_log_panel(col_m)

        # ── COLUMN B: Text→Sign ──────────────────────────────────────────────
        self.col_b = ctk.CTkFrame(body, fg_color=PANEL, corner_radius=14,
                                   border_color=MUTED, border_width=2)
        self.col_b.pack(side="left", fill="both", expand=True,
                        padx=(6,12), pady=12)

        self._build_text_to_sign_panel(self.col_b)

    # ── Sign → Text panel ────────────────────────────────────────────────────

    def _build_sign_to_text_panel(self, parent):
        ctk.CTkLabel(parent, text="SIGN → TEXT",
                     font=("Courier New", 11, "bold"), text_color=ACCENT).pack(
            anchor="nw", padx=14, pady=(10,0))
        ctk.CTkLabel(parent, text="Deaf user signs here",
                     font=("Courier New", 9), text_color=MUTED).pack(anchor="nw", padx=14)

        # Camera feed
        ctk.CTkLabel(parent, text="LIVE CAMERA",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=14, pady=(8,0))
        self.video_lbl = ctk.CTkLabel(parent, text="", fg_color="#0a0a0c", corner_radius=8)
        self.video_lbl.pack(padx=10, pady=4)

        # Skeleton
        sk_row = ctk.CTkFrame(parent, fg_color="transparent")
        sk_row.pack(fill="x", padx=10)
        self.skel_lbl = ctk.CTkLabel(sk_row, text="", fg_color="#0a0a0c", corner_radius=8)
        self.skel_lbl.pack(side="left")

        # Prediction
        pred_card = ctk.CTkFrame(sk_row, fg_color=CARD, corner_radius=10)
        pred_card.pack(side="left", fill="both", expand=True, padx=(8,0))
        ctk.CTkLabel(pred_card, text="DETECTING",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=10, pady=(8,0))
        self.pred_lbl = ctk.CTkLabel(pred_card, text="—",
                                     font=("Courier New", 40, "bold"), text_color=ACCENT)
        self.pred_lbl.pack()
        self.conf_bar = ctk.CTkProgressBar(pred_card, width=150, height=5,
                                            fg_color="#1e1e28", progress_color=ACCENT,
                                            corner_radius=3)
        self.conf_bar.set(0)
        self.conf_bar.pack(pady=(2,4))
        self.conf_lbl = ctk.CTkLabel(pred_card, text="0%",
                                     font=("Courier New", 10), text_color=MUTED)
        self.conf_lbl.pack(pady=(0,8))

        # Debounce
        deb_card = ctk.CTkFrame(parent, fg_color=CARD, corner_radius=10)
        deb_card.pack(fill="x", padx=10, pady=4)
        ctk.CTkLabel(deb_card, text="CONFIRMING SIGN",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=10, pady=(6,0))
        self.debounce_bar = ctk.CTkProgressBar(deb_card, height=7,
                                                fg_color="#1e1e28",
                                                progress_color=ACCENT2, corner_radius=3)
        self.debounce_bar.set(0)
        self.debounce_bar.pack(fill="x", padx=10, pady=(2,2))
        self.debounce_lbl = ctk.CTkLabel(deb_card, text="hold sign to confirm…",
                                          font=("Courier New", 10), text_color=MUTED)
        self.debounce_lbl.pack(pady=(0,6))

        # Sign mode toggle
        mode_row = ctk.CTkFrame(parent, fg_color="transparent")
        mode_row.pack(fill="x", padx=10, pady=4)
        self.btn_letter_mode = ctk.CTkButton(
            mode_row, text="LETTER", width=120, height=34,
            fg_color=ACCENT, hover_color="#00b8d4", text_color=BG,
            font=("Courier New", 11, "bold"), corner_radius=8,
            command=lambda: self._set_sign_mode("letter"))
        self.btn_letter_mode.pack(side="left", padx=(0,6))
        self.btn_word_mode = ctk.CTkButton(
            mode_row, text="WORD", width=120, height=34,
            fg_color=BTN_H, hover_color="#222230", text_color=MUTED,
            font=("Courier New", 11, "bold"), corner_radius=8,
            command=lambda: self._set_sign_mode("word"))
        self.btn_word_mode.pack(side="left")

        # Buffer
        buf_card = ctk.CTkFrame(parent, fg_color=CARD, corner_radius=10)
        buf_card.pack(fill="x", padx=10, pady=4)
        ctk.CTkLabel(buf_card, text="SIGNED MESSAGE",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=10, pady=(6,0))
        self.sign_buffer_lbl = ctk.CTkLabel(buf_card, text="",
                                             font=("Courier New", 18, "bold"),
                                             text_color=TEXT, wraplength=280,
                                             justify="left")
        self.sign_buffer_lbl.pack(padx=10, pady=(0,8), anchor="w")

        # Action buttons
        act_row = ctk.CTkFrame(parent, fg_color="transparent")
        act_row.pack(fill="x", padx=10, pady=(4,10))

        self.btn_send_sign = ctk.CTkButton(
            act_row, text="▶  SEND MESSAGE", height=40,
            fg_color=ACCENT, hover_color="#00b8d4", text_color=BG,
            font=("Courier New", 12, "bold"), corner_radius=8,
            command=self._on_send_signed_message)
        self.btn_send_sign.pack(fill="x", pady=(0,6))

        del_row = ctk.CTkFrame(act_row, fg_color="transparent")
        del_row.pack(fill="x")
        ctk.CTkButton(del_row, text="DEL", height=32, width=100,
                      fg_color=BTN_H, hover_color="#2a1a1a", text_color=ACCENT2,
                      font=("Courier New", 10, "bold"), corner_radius=6,
                      command=self._sign_delete_last).pack(side="left", padx=(0,6))
        ctk.CTkButton(del_row, text="CLEAR", height=32, width=100,
                      fg_color=BTN_H, hover_color="#2a1a1a", text_color=MUTED,
                      font=("Courier New", 10, "bold"), corner_radius=6,
                      command=self._sign_clear).pack(side="left")

    # ── Text → Sign panel ────────────────────────────────────────────────────

    def _build_text_to_sign_panel(self, parent):
        ctk.CTkLabel(parent, text="TEXT → SIGN",
                     font=("Courier New", 11, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=14, pady=(10,0))
        ctk.CTkLabel(parent, text="Hearing user replies here",
                     font=("Courier New", 9), text_color=MUTED).pack(anchor="nw", padx=14)

        # Sign display canvas
        ctk.CTkLabel(parent, text="SIGN DISPLAY",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=14, pady=(8,0))

        self.sign_canvas_frame = ctk.CTkFrame(parent, fg_color="#0a0a0c",
                                               corner_radius=10,
                                               width=CANVAS_W, height=CANVAS_H)
        self.sign_canvas_frame.pack(padx=10, pady=4)
        self.sign_canvas_frame.pack_propagate(False)

        self.sign_display_lbl = ctk.CTkLabel(self.sign_canvas_frame, text="",
                                              font=("Segoe UI Emoji", 64),
                                              text_color=MUTED, fg_color="#0a0a0c")
        self.sign_display_lbl.pack(expand=True)

        # Progress / label
        info_card = ctk.CTkFrame(parent, fg_color=CARD, corner_radius=10)
        info_card.pack(fill="x", padx=10, pady=4)
        self.sign_word_lbl = ctk.CTkLabel(info_card, text="—",
                                          font=("Courier New", 28, "bold"),
                                          text_color=ACCENT2)
        self.sign_word_lbl.pack(pady=(6,2))
        self.sign_note_lbl = ctk.CTkLabel(info_card, text="Enter text below and press Send Reply",
                                          font=("Courier New", 10), text_color=MUTED)
        self.sign_note_lbl.pack(pady=(0,6))
        self.sign_prog_bar = ctk.CTkProgressBar(info_card, height=5,
                                                 fg_color="#1e1e28",
                                                 progress_color=ACCENT2, corner_radius=3)
        self.sign_prog_bar.set(0)
        self.sign_prog_bar.pack(fill="x", padx=10, pady=(0,8))

        # Speed
        spd_row = ctk.CTkFrame(parent, fg_color="transparent")
        spd_row.pack(fill="x", padx=10, pady=4)
        ctk.CTkLabel(spd_row, text="SPEED",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(side="left")
        spd_slider = ctk.CTkSlider(spd_row, from_=0.5, to=4.0,
                                   variable=self._speed_var,
                                   fg_color="#1e1e28", progress_color=ACCENT2,
                                   button_color=ACCENT2, width=140)
        spd_slider.pack(side="left", padx=8)
        self.spd_lbl = ctk.CTkLabel(spd_row, text="1.5×",
                                     font=("Courier New", 10), text_color=MUTED)
        self.spd_lbl.pack(side="left")
        self._speed_var.trace_add("write", lambda *_:
            self.spd_lbl.configure(text=f"{self._speed_var.get():.1f}×"))

        # Input
        ctk.CTkLabel(parent, text="TYPE / SPEAK REPLY",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=14, pady=(8,0))

        inp_frame = ctk.CTkFrame(parent, fg_color="transparent")
        inp_frame.pack(fill="x", padx=10, pady=4)

        self.reply_entry = ctk.CTkEntry(inp_frame,
                                        font=("Courier New", 13),
                                        fg_color=CARD, text_color=TEXT,
                                        placeholder_text="Type your reply…",
                                        border_color=BORDER, corner_radius=8,
                                        height=38)
        self.reply_entry.pack(fill="x", pady=(0,6))
        self.reply_entry.bind("<Return>", lambda e: self._on_send_reply())

        btn_row2 = ctk.CTkFrame(inp_frame, fg_color="transparent")
        btn_row2.pack(fill="x")

        self.btn_send_reply = ctk.CTkButton(
            btn_row2, text="▶  SEND REPLY", height=38, width=160,
            fg_color=MUTED, hover_color=MUTED, text_color=BG,
            font=("Courier New", 12, "bold"), corner_radius=8,
            command=self._on_send_reply)
        self.btn_send_reply.pack(side="left", padx=(0,6))

        self.btn_mic = ctk.CTkButton(
            btn_row2, text="🎤 MIC", height=38, width=90,
            fg_color=BTN_H, hover_color="#222230", text_color=GREEN,
            font=("Courier New", 11, "bold"), corner_radius=8,
            command=self._on_speech_input)
        self.btn_mic.pack(side="left")

        # Letter strip
        self.letter_strip = ctk.CTkFrame(parent, fg_color="transparent")
        self.letter_strip.pack(fill="x", padx=10, pady=4)

    # ── Chat log panel ───────────────────────────────────────────────────────

    def _build_chat_log_panel(self, parent):
        ctk.CTkLabel(parent, text="CONVERSATION",
                     font=("Courier New", 10, "bold"), text_color=MUTED).pack(
            padx=12, pady=(12,4), anchor="nw")

        self.chat_scroll = ctk.CTkScrollableFrame(parent, fg_color=BG,
                                                   corner_radius=8)
        self.chat_scroll.pack(fill="both", expand=True, padx=8, pady=(0,8))

    def _add_chat_bubble(self, role, text):
        """Add a bubble to the chat log."""
        self.chat_log.append({"role": role, "text": text})
        is_deaf = role == "deaf"
        bg  = "#0d2a3a" if is_deaf else "#1a1a0d"
        tc  = ACCENT    if is_deaf else ACCENT2
        tag = "Deaf" if is_deaf else "Hearing"

        bubble = ctk.CTkFrame(self.chat_scroll, fg_color=bg, corner_radius=10)
        bubble.pack(fill="x", pady=4, padx=4,
                    anchor="w" if is_deaf else "e")

        ctk.CTkLabel(bubble, text=tag,
                     font=("Courier New", 8, "bold"), text_color=tc).pack(
            anchor="nw", padx=8, pady=(6,0))
        ctk.CTkLabel(bubble, text=text,
                     font=("Courier New", 11), text_color=TEXT,
                     wraplength=170, justify="left").pack(padx=8, pady=(2,8))

        # Auto-scroll to bottom
        self.root.after(50, lambda: self.chat_scroll._parent_canvas.yview_moveto(1.0))

    # ══════════════════════════════════════════════════════════════════════════
    #  CONVERSATION MODE MANAGEMENT
    # ══════════════════════════════════════════════════════════════════════════

    def _activate_mode(self, mode):
        """Switch visual state to "sign_to_text" or "text_to_sign"."""
        self.conv_mode = mode
        if mode == "sign_to_text":
            # Highlight left panel
            self.col_a.configure(border_color=ACCENT, border_width=2)
            self.col_b.configure(border_color=MUTED,  border_width=2)
            self.mode_badge.configure(
                text="● SIGN → TEXT  (Deaf Turn)", text_color=ACCENT)
            # Enable sign camera controls
            self.btn_send_sign.configure(fg_color=ACCENT, hover_color="#00b8d4",
                                          text_color=BG, state="normal")
            # Dim reply controls
            self.btn_send_reply.configure(fg_color=MUTED, hover_color=MUTED,
                                           text_color=BG, state="disabled")
            self.btn_mic.configure(state="disabled")
        else:
            # Highlight right panel
            self.col_a.configure(border_color=MUTED,  border_width=2)
            self.col_b.configure(border_color=ACCENT2, border_width=2)
            self.mode_badge.configure(
                text="● TEXT → SIGN  (Hearing Turn)", text_color=ACCENT2)
            self.btn_send_sign.configure(fg_color=BTN_H, hover_color=BTN_H,
                                          text_color=MUTED, state="disabled")
            self.btn_send_reply.configure(fg_color=ACCENT2, hover_color="#cc2a55",
                                           text_color=BG, state="normal")
            self.btn_mic.configure(state="normal")

    # ══════════════════════════════════════════════════════════════════════════
    #  FLOW:  SIGN → TEXT
    # ══════════════════════════════════════════════════════════════════════════

    def _on_send_signed_message(self):
        text = self.sign_buffer.strip()
        if not text:
            return

        ConfirmDialog(
            self.root,
            title="Send Signed Message",
            message=f'Signed message:\n\n"{text}"\n\nHow would you like to send it?',
            buttons=[
                ("confirm", "Show Text"),
                ("tts",     "Speak Aloud"),
                ("cancel",  "Cancel"),
            ],
            on_result=lambda choice: self._after_sign_confirm(choice, text),
            accent=ACCENT,
        )

    def _after_sign_confirm(self, choice, text):
        if choice == "cancel":
            return
        if choice == "tts":
            threading.Thread(target=self._tts_run, args=(text,), daemon=True).start()
        # Always add to chat log
        self._add_chat_bubble("deaf", text)
        self.sign_buffer = ""
        self._update_sign_buffer_ui()
        # Ask to switch
        self.root.after(300, lambda: self._ask_switch_to_text_to_sign())

    def _ask_switch_to_text_to_sign(self):
        ConfirmDialog(
            self.root,
            title="Switch Mode",
            message="Message sent!\n\nSwitch to TEXT → SIGN so the hearing person can reply?",
            buttons=[
                ("switch", "⇄  Switch"),
                ("cancel", "Stay Here"),
            ],
            on_result=lambda c: self._activate_mode("text_to_sign") if c == "switch" else None,
            accent=WARN_FG,
        )

    # ══════════════════════════════════════════════════════════════════════════
    #  FLOW:  TEXT → SIGN
    # ══════════════════════════════════════════════════════════════════════════

    def _on_send_reply(self):
        text = self.reply_entry.get().strip()
        if not text:
            return

        ConfirmDialog(
            self.root,
            title="Send Reply as Signs",
            message=f'Reply text:\n\n"{text}"\n\nPlay this as ASL signs for the deaf user?',
            buttons=[
                ("confirm", " Play Signs"),
                ("cancel",  " Cancel"),
            ],
            on_result=lambda choice: self._after_reply_confirm(choice, text),
            accent=ACCENT2,
        )

    def _after_reply_confirm(self, choice, text):
        if choice == "cancel":
            return
        self._add_chat_bubble("hearing", text)
        self.reply_entry.delete(0, "end")
        # Start building & playing signs
        self.sign_word_lbl.configure(text="⏳")
        self.sign_note_lbl.configure(text="Loading signs…")
        threading.Thread(target=self._load_and_play_signs,
                         args=(text,), daemon=True).start()

    def _load_and_play_signs(self, text):
        seq = build_sign_sequence(text)
        self.root.after(0, lambda: self._start_sign_playback(seq))

    def _start_sign_playback(self, seq):
        self._stop_sign_playback()
        self.sign_sequence  = seq
        self.sign_seq_idx   = 0
        self.sign_playing   = True
        self.sign_prog_bar.set(0)
        self._advance_sign()

    def _advance_sign(self):
        if not self.sign_playing:
            return
        if self.sign_seq_idx >= len(self.sign_sequence):
            self.sign_playing = False
            self.sign_word_lbl.configure(text="✓")
            self.sign_note_lbl.configure(text="Done!")
            self.sign_prog_bar.set(1.0)
            # Ask to switch back
            self.root.after(600, self._ask_switch_to_sign_to_text)
            return

        item = self.sign_sequence[self.sign_seq_idx]
        pct  = (self.sign_seq_idx + 1) / max(len(self.sign_sequence), 1)
        self.sign_prog_bar.set(pct)

        if item["kind"] == "space":
            self.sign_seq_idx += 1
            self.sign_gif_job = self.root.after(
                int(250 / self._speed_var.get()), self._advance_sign)
            return

        self.sign_word_lbl.configure(text=item["label"])
        kind = item["kind"]
        note = "Word sign (GIF)" if kind == "gif" else "Fingerspelling"
        self.sign_note_lbl.configure(text=f"{note}: {item['label']}")

        frames = item.get("frames", [])
        if not frames:
            self.sign_seq_idx += 1
            self.sign_gif_job = self.root.after(200, self._advance_sign)
            return

        if kind == "gif" and len(frames) > 1:
            self._play_gif_frames(frames, 0)
        else:
            self._show_sign_frame(frames[0])
            hold_ms = int(900 / self._speed_var.get())
            self.sign_seq_idx += 1
            self.sign_gif_job = self.root.after(hold_ms, self._advance_sign)

    def _play_gif_frames(self, frames, frame_idx):
        if not self.sign_playing:
            return
        self._show_sign_frame(frames[frame_idx])
        nxt = frame_idx + 1
        if nxt < len(frames):
            delay = max(40, int(60 / self._speed_var.get()))
            self.sign_gif_job = self.root.after(delay,
                lambda: self._play_gif_frames(frames, nxt))
        else:
            self.sign_seq_idx += 1
            self.sign_gif_job = self.root.after(300, self._advance_sign)

    def _show_sign_frame(self, img: Image.Image):
        ph = ImageTk.PhotoImage(img)
        self._static_img_ref = ph
        self.sign_display_lbl.configure(image=ph, text="")
        self.sign_display_lbl._img = ph

    def _stop_sign_playback(self):
        self.sign_playing = False
        if self.sign_gif_job:
            self.root.after_cancel(self.sign_gif_job)
            self.sign_gif_job = None

    def _ask_switch_to_sign_to_text(self):
        ConfirmDialog(
            self.root,
            title="Switch Mode",
            message="Signs played!\n\nSwitch back to SIGN → TEXT so the deaf person can reply?",
            buttons=[
                ("switch", "⇄  Switch"),
                ("cancel", "Stay Here"),
            ],
            on_result=lambda c: self._activate_mode("sign_to_text") if c == "switch" else None,
            accent=WARN_FG,
        )

    # ══════════════════════════════════════════════════════════════════════════
    #  SPEECH INPUT  (for hearing user)
    # ══════════════════════════════════════════════════════════════════════════

    def _on_speech_input(self):
        self.btn_mic.configure(text="  Listening…", text_color="#ff3d71")
        threading.Thread(target=self._listen_mic, daemon=True).start()

    def _listen_mic(self):
        r = sr.Recognizer()
        try:
            with sr.Microphone() as src:
                r.adjust_for_ambient_noise(src, duration=0.5)
                audio = r.listen(src, timeout=6, phrase_time_limit=8)
            text = r.recognize_google(audio)
            self.root.after(0, lambda: self.reply_entry.delete(0, "end"))
            self.root.after(0, lambda: self.reply_entry.insert(0, text))
        except sr.WaitTimeoutError:
            pass
        except sr.UnknownValueError:
            pass
        except Exception:
            pass
        finally:
            self.root.after(0, lambda: self.btn_mic.configure(
                text="🎤 MIC", text_color=GREEN))

    # ══════════════════════════════════════════════════════════════════════════
    #  MODEL LOADING
    # ══════════════════════════════════════════════════════════════════════════

    def _load_models_async(self):
        threading.Thread(target=self._load_models, daemon=True).start()

    def _load_models(self):
        try:
            self.letter_model = tf.keras.models.load_model(MODEL_LETTER)
            self.root.after(0, lambda: self.status_dot.configure(
                text="● LETTER ✓  WORD…", text_color="#ffaa00"))
        except Exception as e:
            self.root.after(0, lambda: self.status_dot.configure(
                text=f"⚠ LETTER: {e}", text_color=ACCENT2))

        try:
            self.word_model = tf.keras.models.load_model(MODEL_WORD)
            self.root.after(0, lambda: self.status_dot.configure(
                text="● READY", text_color=ACCENT))
        except Exception as e:
            self.root.after(0, lambda: self.status_dot.configure(
                text=f"⚠ WORD: {e}", text_color=ACCENT2))

        self.holistic      = mp_holistic.Holistic(min_detection_confidence=0.5,
                                                   min_tracking_confidence=0.5)
        self.hand_detector = mp_hands_mod.Hands(min_detection_confidence=0.5,
                                                 min_tracking_confidence=0.5,
                                                 max_num_hands=1)
        # Start in sign-to-text mode
        self.root.after(0, lambda: self._activate_mode("sign_to_text"))

    # ══════════════════════════════════════════════════════════════════════════
    #  CAMERA LOOP
    # ══════════════════════════════════════════════════════════════════════════

    def _camera_loop(self):
        if not self.running:
            return
        ret, frame = self.cap.read()
        if ret:
            frame = cv2.flip(frame, 1)
            skel  = np.zeros((224,224,3), dtype=np.uint8)
            rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            if self.sign_mode == "letter":
                self._process_letter(rgb, skel)
            else:
                self._process_word(rgb)

            # Draw overlay badge
            h, w = frame.shape[:2]
            tag  = "LETTER" if self.sign_mode == "letter" else "WORD"
            col  = (0,229,255) if self.sign_mode == "letter" else (255,61,113)
            cv2.rectangle(frame, (w-110, 0), (w, 28), (13,13,15), -1)
            cv2.putText(frame, tag, (w-100, 20),
                        cv2.FONT_HERSHEY_DUPLEX, 0.65, col, 1, cv2.LINE_AA)

            # Video display
            scale = 400 / frame.shape[1]
            disp  = cv2.resize(rgb, (400, int(frame.shape[0]*scale)))
            img   = Image.fromarray(disp)
            cimg  = ctk.CTkImage(light_image=img, dark_image=img, size=(400, int(frame.shape[0]*scale)))
            self.video_lbl.configure(image=cimg)
            self.video_lbl._image = cimg

            # Skeleton
            skel_rgb = cv2.cvtColor(self.skeleton_frame, cv2.COLOR_BGR2RGB)
            skel_img = Image.fromarray(skel_rgb)
            cskel    = ctk.CTkImage(light_image=skel_img, dark_image=skel_img, size=(120,120))
            self.skel_lbl.configure(image=cskel)
            self.skel_lbl._image = cskel

        self.root.after(15, self._camera_loop)

    # ══════════════════════════════════════════════════════════════════════════
    #  SIGN DETECTION  (letter + word)
    # ══════════════════════════════════════════════════════════════════════════

    def _process_letter(self, rgb, skel):
        if self.hand_detector is None or self.letter_model is None:
            return
        result = self.hand_detector.process(rgb)
        if result.multi_hand_landmarks:
            for hand in result.multi_hand_landmarks:
                pts = normalize_landmarks(hand.landmark, 224, 224)
                draw_hand_skeleton(skel, pts)
            self.skeleton_frame = skel
            inp  = tf.keras.applications.resnet50.preprocess_input(
                np.expand_dims(skel, axis=0).astype(np.float32))
            pred = self.letter_model.predict(inp, verbose=0)[0]
            idx  = int(np.argmax(pred)); conf = float(pred[idx])
            label = self.LETTER_LABELS[idx]
            self.cur_prediction = label; self.cur_confidence = conf
            self._update_pred_ui(label, conf)
            if conf >= self.CONF_THRESH and label not in ('ok',):
                now = time.time()
                if self.pending_letter != label:
                    self.pending_letter = label; self.pending_since = now
                elapsed  = now - (self.pending_since or now)
                progress = min(elapsed / self.DEBOUNCE_SEC, 1.0)
                self.debounce_bar.set(progress)
                self.debounce_lbl.configure(
                    text=f'confirming "{label}"… {int(progress*100)}%')
                if elapsed >= self.DEBOUNCE_SEC:
                    self._confirm_letter(label)
                    self.pending_letter = None; self.pending_since = None
                    self.debounce_bar.set(0)
                    self.debounce_lbl.configure(text="hold sign to confirm…")
            else:
                self.pending_letter = None; self.pending_since = None
                self.debounce_bar.set(0)

    def _confirm_letter(self, letter):
        if letter == 'del':
            self.sign_buffer = self.sign_buffer[:-1]
        elif letter == 'space':
            self.sign_buffer += ' '
        else:
            self.sign_buffer += letter
        self._update_sign_buffer_ui()

    def _process_word(self, rgb):
        if self.holistic is None or self.word_model is None:
            return
        img_rgb = rgb.copy(); img_rgb.flags.writeable = False
        result  = self.holistic.process(img_rgb)
        kp = extract_keypoints(result)
        self.word_sequence.append(kp)
        if len(self.word_sequence) == 30:
            seq  = np.expand_dims(np.array(self.word_sequence), axis=0)
            pred = self.word_model.predict(seq, verbose=0)[0]
            idx  = int(np.argmax(pred)); conf = float(pred[idx])
            label = self.WORD_LABELS[idx]
            self._update_pred_ui(label, conf)
            if conf >= self.CONF_THRESH:
                if idx == self.last_word_pred:
                    self.word_stab_count += 1
                else:
                    self.last_word_pred = idx; self.word_stab_count = 1
                if (self.word_stab_count >= self.WORD_STAB_FRM
                        and label != 'nothing'
                        and label != self.last_confirmed_word):
                    self.sign_buffer += label + ' '
                    self.last_confirmed_word = label
                    self.word_stab_count = 0
                    self._update_sign_buffer_ui()
            else:
                self.last_word_pred = None; self.word_stab_count = 0

    # ══════════════════════════════════════════════════════════════════════════
    #  SIGN MODE TOGGLE
    # ══════════════════════════════════════════════════════════════════════════

    def _set_sign_mode(self, mode):
        self.sign_mode = mode
        if mode == "letter":
            self.btn_letter_mode.configure(fg_color=ACCENT, text_color=BG)
            self.btn_word_mode.configure(fg_color=BTN_H,  text_color=MUTED)
        else:
            self.btn_word_mode.configure(fg_color=ACCENT, text_color=BG)
            self.btn_letter_mode.configure(fg_color=BTN_H, text_color=MUTED)
        self.pending_letter = None; self.pending_since = None
        self.word_sequence.clear(); self.word_stab_count = 0
        self.last_word_pred = None; self.last_confirmed_word = None

    # ══════════════════════════════════════════════════════════════════════════
    #  UI UPDATE HELPERS
    # ══════════════════════════════════════════════════════════════════════════

    def _update_pred_ui(self, label, conf):
        self.pred_lbl.configure(text=label.upper())
        self.conf_bar.set(conf)
        self.conf_lbl.configure(text=f"{int(conf*100)}%")

    def _update_sign_buffer_ui(self):
        self.sign_buffer_lbl.configure(
            text=self.sign_buffer if self.sign_buffer else "")

    def _sign_delete_last(self):
        self.sign_buffer = self.sign_buffer.rstrip()
        if ' ' in self.sign_buffer:
            self.sign_buffer = self.sign_buffer.rsplit(' ', 1)[0] + ' '
        else:
            self.sign_buffer = self.sign_buffer[:-1]
        self._update_sign_buffer_ui()

    def _sign_clear(self):
        self.sign_buffer = ""
        self._update_sign_buffer_ui()
        self.pending_letter = None; self.pending_since = None
        self.debounce_bar.set(0)

    def _tts_run(self, text):
        self.tts_engine.say(text)
        self.tts_engine.runAndWait()

    # ══════════════════════════════════════════════════════════════════════════
    #  CLOSE
    # ══════════════════════════════════════════════════════════════════════════

    def on_close(self):
        self.running = False
        self._stop_sign_playback()
        self.cap.release()
        if self.holistic:
            self.holistic.close()
        if self.hand_detector:
            self.hand_detector.close()
        self.root.destroy()


# ──────────────────────────────────────────────────────────────────────────────
#  ENTRY POINT
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    ctk.set_appearance_mode("dark")
    ctk.set_default_color_theme("blue")

    root = ctk.CTk()
    app  = ASLBidirectionalApp(root)
    root.protocol("WM_DELETE_WINDOW", app.on_close)
    root.mainloop()

2026-04-12 17:03:45.318864: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-12 17:03:45.455976: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-12 17:03:45.456005: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-12 17:03:45.460615: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-12 17:03:45.477838: I tensorflow/core/platform/cpu_feature_guar

In [2]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║         ASL BIDIRECTIONAL CONVERSATION SYSTEM  (FULLY OFFLINE)       ║
║                                                                      ║
║  MODE A  ─  Sign → Text / TTS                                        ║
║    Deaf/HoH user signs via camera; ResNet (letter) + LSTM (word)     ║
║    detects signs, fills sentence buffer, then confirms to speak/show ║
║                                                                      ║
║  MODE B  ─  Text → Sign                                              ║
║    Hearing user's reply typed/spoken; system plays ASL GIF / finger- ║
║    spelling so the deaf user can read the response                   ║
║                                                                      ║
║  FLOW  ─  Alternating conversation turns with confirm dialogs:       ║
║    [Sign→Text] → confirm → speak/show text → switch alert            ║
║    [Text→Sign] → confirm → play ASL signs → switch alert → repeat    ║
║                                                                      ║
║  Requirements:                                                       ║
║    pip install customtkinter opencv-python mediapipe tensorflow      ║
║               pyttsx3 pillow speechrecognition                       ║
║                                                                      ║
║  NOTE: 100% offline — no internet connection required.               ║
╚══════════════════════════════════════════════════════════════════════╝
"""

import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp
from collections import deque
import pyttsx3
import time
import threading
import tkinter as tk
from tkinter import ttk
import customtkinter as ctk
from PIL import Image, ImageTk, ImageDraw, ImageFont, ImageSequence
import os
import speech_recognition as sr

# ──────────────────────────────────────────────────────────────────────────────
#  PATHS  ── edit to match your system
# ──────────────────────────────────────────────────────────────────────────────

MODEL_LETTER  = '/home/maries/Development/Project_OWSLT/models/letter_modelV2.keras'
MODEL_WORD    = '/home/maries/Development/Project_OWSLT/models/word_modelV2.keras'
LETTER_FOLDER = '/home/maries/Development/Project_OWSLT/assets/ASl letters/'
GIF_FOLDER    = '/home/maries/Development/Project_OWSLT/assets/ASL gif/'

# ──────────────────────────────────────────────────────────────────────────────
#  PALETTE
# ──────────────────────────────────────────────────────────────────────────────

BG       = "#0d0d0f"
PANEL    = "#16161a"
CARD     = "#1c1c22"
BORDER   = "#2a2a35"
ACCENT   = "#00e5ff"    # sign→text  (cyan)
ACCENT2  = "#ff3d71"    # text→sign  (rose)
GREEN    = "#00ff9d"
MUTED    = "#6b6b80"
TEXT     = "#e8e8f0"
WHITE    = "#ffffff"
BTN_H    = "#1a1a22"
WARN_BG  = "#1a1200"
WARN_FG  = "#ffcc00"

# ──────────────────────────────────────────────────────────────────────────────
#  ASL HAND SKELETON HELPERS
# ──────────────────────────────────────────────────────────────────────────────

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),
]

FINGER_COLORS = {
    (0,1):(0,255,255),(1,2):(0,255,255),(2,3):(0,255,255),(3,4):(0,255,255),
    (0,5):(255,80,80),(5,6):(255,80,80),(6,7):(255,80,80),(7,8):(255,80,80),
    (0,9):(80,255,80),(9,10):(80,255,80),(10,11):(80,255,80),(11,12):(80,255,80),
    (0,13):(80,80,255),(13,14):(80,80,255),(14,15):(80,80,255),(15,16):(80,80,255),
    (0,17):(255,80,255),(17,18):(255,80,255),(18,19):(255,80,255),(19,20):(255,80,255),
    (5,9):(220,220,220),(9,13):(220,220,220),(13,17):(220,220,220),
}

def normalize_landmarks(landmarks, img_w, img_h):
    points = np.array([[lm.x * img_w, lm.y * img_h] for lm in landmarks])
    mn = points.min(axis=0); mx = points.max(axis=0)
    cx = (mn[0]+mx[0])/2; cy = (mn[1]+mx[1])/2
    dim = max(mx[0]-mn[0], mx[1]-mn[1])
    scale = (min(img_w,img_h)*0.8)/dim if dim > 0 else 1
    return [(int((p[0]-cx)*scale+img_w/2), int((p[1]-cy)*scale+img_h/2)) for p in points]

def draw_hand_skeleton(frame, pts):
    for c in HAND_CONNECTIONS:
        cv2.line(frame, pts[c[0]], pts[c[1]], FINGER_COLORS[c], 3)
    for i, p in enumerate(pts):
        color = (0,0,255) if i in [4,8,12,16,20] else (255,0,255)
        cv2.circle(frame, p, 3, color, -1)
        cv2.circle(frame, p, 3, (255,255,255), 1)
    return frame

mp_holistic   = mp.solutions.holistic
mp_hands_mod  = mp.solutions.hands

def extract_keypoints(results):
    pose = (np.array([[r.x,r.y,r.z,r.visibility] for r in results.pose_landmarks.landmark]).flatten()
            if results.pose_landmarks else np.zeros(132))
    lh   = (np.array([[r.x,r.y,r.z] for r in results.left_hand_landmarks.landmark]).flatten()
            if results.left_hand_landmarks else np.zeros(63))
    rh   = (np.array([[r.x,r.y,r.z] for r in results.right_hand_landmarks.landmark]).flatten()
            if results.right_hand_landmarks else np.zeros(63))
    return np.concatenate([pose, lh, rh])

# ──────────────────────────────────────────────────────────────────────────────
#  TEXT → SIGN  HELPERS  (fully local / offline)
# ──────────────────────────────────────────────────────────────────────────────

# Words that have a local GIF available in GIF_FOLDER
GIF_WORDS = {
    "HELLO","THANKS","WATER","GO","UNDERSTAND","EAT","FRIEND",
    "HELP","GOODBYE","MORNING","GOODNIGHT","I","PLEASE","SORRY",
    "YES","YOU","FOOD","SAD","GOODEVENING","DRINK","NO",
}

# Aliases: normalise multi-word phrases → single GIF filename key
WORD_ALIASES = {
    "good evening": "goodevening",
    "good night":   "goodnight",
    "good morning": "morning",
    "good bye":     "goodbye",
    "thank you":    "thanks",
    "thank":        "thanks",
}

CANVAS_W, CANVAS_H = 460, 340


def fit_image(img: Image.Image, w: int, h: int) -> Image.Image:
    """Resize image to fit within (w, h) keeping aspect ratio, centered on dark canvas."""
    img.thumbnail((w, h), Image.LANCZOS)
    canvas = Image.new("RGBA", (w, h), (13, 13, 15, 255))
    x = (w - img.width) // 2
    y = (h - img.height) // 2
    canvas.paste(img, (x, y), img if img.mode == "RGBA" else None)
    return canvas


def make_placeholder(letter: str) -> Image.Image:
    """
    Create a simple dark placeholder image showing the letter when no
    local image file is found.  No network access — purely drawn with Pillow.
    """
    img  = Image.new("RGBA", (CANVAS_W, CANVAS_H), (13, 13, 15, 255))
    draw = ImageDraw.Draw(img)

    # Try to load a nice font; fall back gracefully
    try:
        fnt_big = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 110)
        fnt_sub = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 20)
    except Exception:
        fnt_big = ImageFont.load_default()
        fnt_sub = fnt_big

    # Centre the big letter
    bb  = draw.textbbox((0, 0), letter, font=fnt_big)
    tw, th = bb[2] - bb[0], bb[3] - bb[1]
    draw.text(((CANVAS_W - tw) // 2, (CANVAS_H - th) // 2 - 20),
              letter, fill=(0, 229, 255, 220), font=fnt_big)

    # Small subtitle
    sub = f"ASL  {letter}  (image not found)"
    bb2 = draw.textbbox((0, 0), sub, font=fnt_sub)
    draw.text(((CANVAS_W - (bb2[2] - bb2[0])) // 2, CANVAS_H - 50),
              sub, fill=(107, 107, 128, 200), font=fnt_sub)

    return img


def find_gif_local(word: str):
    """
    Return the local filesystem path to a GIF for *word*, or None.
    Checks GIF_WORDS set first, then does a case-insensitive scan of GIF_FOLDER
    as a fallback (so files not listed in GIF_WORDS but present on disk still work).
    """
    w = word.upper().strip()
    # Apply alias mapping
    w = WORD_ALIASES.get(w.lower(), w)

    # 1) Fast check via known set
    if w in GIF_WORDS:
        for ext in (".gif", ".GIF"):
            p = os.path.join(GIF_FOLDER, w + ext)
            if os.path.exists(p):
                return p

    # 2) Slow fallback: scan directory for any case variant
    if os.path.isdir(GIF_FOLDER):
        target = w + ".gif"
        for fname in os.listdir(GIF_FOLDER):
            if fname.upper() == target.upper():
                return os.path.join(GIF_FOLDER, fname)

    return None


def find_letter_image_local(letter: str):
    """
    Return the local filesystem path to a static image for *letter*, or None.
    Searches LETTER_FOLDER for common image extensions, case-insensitively.
    """
    l = letter.upper()
    # Try explicit extension combinations first (fast)
    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG",
                ".bmp", ".BMP", ".webp", ".WEBP"):
        p = os.path.join(LETTER_FOLDER, l + ext)
        if os.path.exists(p):
            return p

    # Slow fallback: scan the folder
    if os.path.isdir(LETTER_FOLDER):
        for fname in os.listdir(LETTER_FOLDER):
            name, ext = os.path.splitext(fname)
            if name.upper() == l and ext.lower() in (".png", ".jpg", ".jpeg",
                                                       ".bmp", ".webp"):
                return os.path.join(LETTER_FOLDER, fname)

    return None


def build_sign_sequence(text: str) -> list:
    """
    Build a list of sign items from *text* using ONLY local assets.

    Each item is a dict:
      { "kind": "gif"|"letter"|"space",
        "label": str,
        "frames": [PIL.Image, ...] }

    Priority per word:
      1. Local GIF  (GIF_FOLDER)
      2. Letter-by-letter fingerspelling from LETTER_FOLDER
      3. Drawn placeholder (if no image file exists for a letter)

    No network calls are made anywhere in this function.
    """
    words  = text.strip().upper().split()
    result = []

    for word in words:
        # ── Apply alias before GIF lookup ─────────────────────────────────
        lookup = WORD_ALIASES.get(word.lower(), word)

        # ── 1. Local GIF ──────────────────────────────────────────────────
        gif_path = find_gif_local(lookup)
        if gif_path:
            try:
                img    = Image.open(gif_path)
                frames = [fit_image(f.copy().convert("RGBA"), CANVAS_W, CANVAS_H)
                          for f in ImageSequence.Iterator(img)]
                if frames:
                    result.append({
                        "kind":   "gif",
                        "label":  word,
                        "frames": frames,
                    })
                    result.append({"kind": "space", "label": "", "frames": []})
                    continue          # done with this word
            except Exception:
                pass                  # corrupt GIF → fall through to fingerspelling

        # ── 2. Fingerspell letter-by-letter ───────────────────────────────
        for ch in word:
            if not ch.isalpha():
                continue

            img_path = find_letter_image_local(ch)
            if img_path:
                try:
                    pil_img = Image.open(img_path).convert("RGBA")
                    frame   = fit_image(pil_img, CANVAS_W, CANVAS_H)
                    result.append({
                        "kind":   "letter",
                        "label":  ch,
                        "frames": [frame],
                    })
                    continue
                except Exception:
                    pass              # unreadable file → use placeholder

            # ── 3. Drawn placeholder ──────────────────────────────────────
            result.append({
                "kind":   "letter",
                "label":  ch,
                "frames": [make_placeholder(ch)],
            })

        result.append({"kind": "space", "label": "", "frames": []})

    return result


# ──────────────────────────────────────────────────────────────────────────────
#  CUSTOM CONFIRM DIALOG  (dark themed, non-blocking via after())
# ──────────────────────────────────────────────────────────────────────────────

class ConfirmDialog(ctk.CTkToplevel):
    """
    Shows a styled modal dialog.
    on_result(choice) is called with "confirm", "tts", or "cancel".
    """
    def __init__(self, parent, title, message, buttons, on_result, accent=ACCENT):
        super().__init__(parent)
        self.on_result = on_result
        self.title(title)
        self.configure(fg_color=PANEL)
        self.resizable(False, False)
        self.grab_set()
        self.attributes("-topmost", True)

        # ── Icon + title ──
        hdr = ctk.CTkFrame(self, fg_color=BG, corner_radius=0)
        hdr.pack(fill="x")
        ctk.CTkLabel(hdr, text=f"◈  {title}",
                     font=("Courier New", 15, "bold"), text_color=accent).pack(
            padx=20, pady=14, anchor="w")

        # ── Message ──
        ctk.CTkLabel(self, text=message,
                     font=("Courier New", 12), text_color=TEXT,
                     wraplength=380, justify="left").pack(padx=24, pady=(8, 20))

        # ── Buttons ──
        btn_row = ctk.CTkFrame(self, fg_color="transparent")
        btn_row.pack(padx=20, pady=(0, 20))

        COLORS = {
            "confirm": (accent,  "#0d0d0f"),
            "tts":     (GREEN,   "#0d0d0f"),
            "switch":  (WARN_FG, "#0d0d0f"),
            "cancel":  (BTN_H,   MUTED),
        }

        for key, label in buttons:
            fg, tc = COLORS.get(key, (BTN_H, TEXT))
            ctk.CTkButton(
                btn_row, text=label, width=140, height=40,
                fg_color=fg, hover_color=fg, text_color=tc,
                font=("Courier New", 12, "bold"), corner_radius=8,
                command=lambda k=key: self._respond(k)
            ).pack(side="left", padx=6)

        self._center(parent)

    def _center(self, parent):
        self.update_idletasks()
        pw = parent.winfo_width();  ph = parent.winfo_height()
        px = parent.winfo_rootx(); py = parent.winfo_rooty()
        w  = self.winfo_reqwidth(); h  = self.winfo_reqheight()
        self.geometry(f"+{px + (pw-w)//2}+{py + (ph-h)//2}")

    def _respond(self, choice):
        self.grab_release()
        self.destroy()
        self.on_result(choice)


# ──────────────────────────────────────────────────────────────────────────────
#  MAIN APPLICATION
# ──────────────────────────────────────────────────────────────────────────────

class ASLBidirectionalApp:
    """
    Two-panel application:
      LEFT  → Sign-to-Text  (camera + ML models)  [MODE A]
      RIGHT → Text-to-Sign  (GIF / fingerspelling) [MODE B]

    Conversation flow (alternates):
      1. Deaf user signs message into buffer  (MODE A active)
      2. Buffer fills → user clicks "Send Message"
         → ConfirmDialog: [Send as Text] [Speak Aloud] [Cancel]
      3. Message displayed in chat log
         → ConfirmDialog: "Switch to Text→Sign so hearing person can reply?"
           [Switch] [Cancel]
      4. Hearing person types / speaks reply  (MODE B active)
      5. User clicks "Send Reply"
         → ConfirmDialog: [Play Signs] [Cancel]
      6. Signs play out
         → ConfirmDialog: "Switch back to Sign→Text so deaf person can reply?"
           [Switch] [Cancel]
      7. Repeat
    """

    LETTER_LABELS = ['a','b','c','d','e','f','g','h','i','j','k','l','m',
                     'n','o','p','q','r','s','t','u','v','w','x','y','z',
                     'del','ok','space']
    WORD_LABELS   = ['hello','help','meet','nice','nothing','please','thanks']

    DEBOUNCE_SEC  = 1.5
    WORD_STAB_FRM = 8
    CONF_THRESH   = 0.70

    def __init__(self, root: ctk.CTk):
        self.root    = root
        self.root.title("ASL Bidirectional Conversation System")
        self.root.geometry("1400x860")
        self.root.resizable(False, False)

        # ── conversation mode: "sign_to_text" | "text_to_sign"
        self.conv_mode = "sign_to_text"

        # ── sign→text state ──
        self.sign_buffer         = ""
        self.running             = True
        self.sign_mode           = "letter"
        self.pending_letter      = None
        self.pending_since       = None
        self.word_sequence       = deque(maxlen=30)
        self.last_word_pred      = None
        self.word_stab_count     = 0
        self.last_confirmed_word = None
        self.cur_prediction      = ""
        self.cur_confidence      = 0.0
        self.skeleton_frame      = np.zeros((224, 224, 3), dtype=np.uint8)

        # ── text→sign state ──
        self.sign_sequence    = []
        self.sign_seq_idx     = 0
        self.sign_playing     = False
        self.sign_gif_job     = None
        self._static_img_ref  = None
        self._speed_var       = tk.DoubleVar(value=1.5)

        # ── shared ──
        self.chat_log = []   # list of {"role": "deaf"|"hearing", "text": str}

        # ── TTS ──
        self.tts_engine = pyttsx3.init()
        self.tts_engine.setProperty('rate', 150)

        # ── Models ──
        self.letter_model  = None
        self.word_model    = None
        self.holistic      = None
        self.hand_detector = None

        self._build_ui()
        self._load_models_async()

        self.cap = cv2.VideoCapture(0)
        self._camera_loop()

    # ══════════════════════════════════════════════════════════════════════════
    #  UI
    # ══════════════════════════════════════════════════════════════════════════

    def _build_ui(self):
        self.root.configure(fg_color=BG)

        # ── top bar ──────────────────────────────────────────────────────────
        topbar = ctk.CTkFrame(self.root, fg_color=PANEL, height=56, corner_radius=0)
        topbar.pack(fill="x", side="top")
        topbar.pack_propagate(False)

        ctk.CTkLabel(topbar, text="◈  ASL BIDIRECTIONAL CONVERSATION",
                     font=("Courier New", 16, "bold"), text_color=ACCENT).pack(
            side="left", padx=20, pady=16)

        self.mode_badge = ctk.CTkLabel(topbar,
            text="● SIGN → TEXT  (Deaf Turn)",
            font=("Courier New", 12, "bold"), text_color=ACCENT)
        self.mode_badge.pack(side="right", padx=24)

        self.status_dot = ctk.CTkLabel(topbar, text="LOADING",
                                       font=("Courier New", 11),
                                       text_color="#ffaa00")
        self.status_dot.pack(side="right", padx=16)

        # ── main body ─────────────────────────────────────────────────────────
        body = ctk.CTkFrame(self.root, fg_color=BG)
        body.pack(fill="both", expand=True)

        # ── COLUMN A: Sign→Text ──────────────────────────────────────────────
        self.col_a = ctk.CTkFrame(body, fg_color=PANEL, corner_radius=14,
                                   border_color=ACCENT, border_width=2)
        self.col_a.pack(side="left", fill="both", expand=True,
                        padx=(12, 6), pady=12)
        self._build_sign_to_text_panel(self.col_a)

        # ── COLUMN M: Chat log (middle) ──────────────────────────────────────
        col_m = ctk.CTkFrame(body, fg_color=CARD, corner_radius=14, width=220)
        col_m.pack(side="left", fill="both", padx=6, pady=12)
        col_m.pack_propagate(False)
        self._build_chat_log_panel(col_m)

        # ── COLUMN B: Text→Sign ──────────────────────────────────────────────
        self.col_b = ctk.CTkFrame(body, fg_color=PANEL, corner_radius=14,
                                   border_color=MUTED, border_width=2)
        self.col_b.pack(side="left", fill="both", expand=True,
                        padx=(6, 12), pady=12)
        self._build_text_to_sign_panel(self.col_b)

    # ── Sign → Text panel ────────────────────────────────────────────────────

    def _build_sign_to_text_panel(self, parent):
        ctk.CTkLabel(parent, text="SIGN → TEXT",
                     font=("Courier New", 11, "bold"), text_color=ACCENT).pack(
            anchor="nw", padx=14, pady=(10, 0))
        ctk.CTkLabel(parent, text="Deaf user signs here",
                     font=("Courier New", 9), text_color=MUTED).pack(
            anchor="nw", padx=14)

        # Camera feed
        ctk.CTkLabel(parent, text="LIVE CAMERA",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=14, pady=(8, 0))
        self.video_lbl = ctk.CTkLabel(parent, text="",
                                      fg_color="#0a0a0c", corner_radius=8)
        self.video_lbl.pack(padx=10, pady=4)

        # Skeleton
        sk_row = ctk.CTkFrame(parent, fg_color="transparent")
        sk_row.pack(fill="x", padx=10)
        self.skel_lbl = ctk.CTkLabel(sk_row, text="",
                                     fg_color="#0a0a0c", corner_radius=8)
        self.skel_lbl.pack(side="left")

        # Prediction
        pred_card = ctk.CTkFrame(sk_row, fg_color=CARD, corner_radius=10)
        pred_card.pack(side="left", fill="both", expand=True, padx=(8, 0))
        ctk.CTkLabel(pred_card, text="DETECTING",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=10, pady=(8, 0))
        self.pred_lbl = ctk.CTkLabel(pred_card, text="—",
                                     font=("Courier New", 40, "bold"),
                                     text_color=ACCENT)
        self.pred_lbl.pack()
        self.conf_bar = ctk.CTkProgressBar(pred_card, width=150, height=5,
                                            fg_color="#1e1e28",
                                            progress_color=ACCENT,
                                            corner_radius=3)
        self.conf_bar.set(0)
        self.conf_bar.pack(pady=(2, 4))
        self.conf_lbl = ctk.CTkLabel(pred_card, text="0%",
                                     font=("Courier New", 10), text_color=MUTED)
        self.conf_lbl.pack(pady=(0, 8))

        # Debounce
        deb_card = ctk.CTkFrame(parent, fg_color=CARD, corner_radius=10)
        deb_card.pack(fill="x", padx=10, pady=4)
        ctk.CTkLabel(deb_card, text="CONFIRMING SIGN",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=10, pady=(6, 0))
        self.debounce_bar = ctk.CTkProgressBar(deb_card, height=7,
                                                fg_color="#1e1e28",
                                                progress_color=ACCENT2,
                                                corner_radius=3)
        self.debounce_bar.set(0)
        self.debounce_bar.pack(fill="x", padx=10, pady=(2, 2))
        self.debounce_lbl = ctk.CTkLabel(deb_card,
                                          text="hold sign to confirm…",
                                          font=("Courier New", 10),
                                          text_color=MUTED)
        self.debounce_lbl.pack(pady=(0, 6))

        # Sign mode toggle
        mode_row = ctk.CTkFrame(parent, fg_color="transparent")
        mode_row.pack(fill="x", padx=10, pady=4)
        self.btn_letter_mode = ctk.CTkButton(
            mode_row, text="LETTER", width=120, height=34,
            fg_color=ACCENT, hover_color="#00b8d4", text_color=BG,
            font=("Courier New", 11, "bold"), corner_radius=8,
            command=lambda: self._set_sign_mode("letter"))
        self.btn_letter_mode.pack(side="left", padx=(0, 6))
        self.btn_word_mode = ctk.CTkButton(
            mode_row, text="WORD", width=120, height=34,
            fg_color=BTN_H, hover_color="#222230", text_color=MUTED,
            font=("Courier New", 11, "bold"), corner_radius=8,
            command=lambda: self._set_sign_mode("word"))
        self.btn_word_mode.pack(side="left")

        # Buffer
        buf_card = ctk.CTkFrame(parent, fg_color=CARD, corner_radius=10)
        buf_card.pack(fill="x", padx=10, pady=4)
        ctk.CTkLabel(buf_card, text="SIGNED MESSAGE",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=10, pady=(6, 0))
        self.sign_buffer_lbl = ctk.CTkLabel(buf_card, text="",
                                             font=("Courier New", 18, "bold"),
                                             text_color=TEXT, wraplength=280,
                                             justify="left")
        self.sign_buffer_lbl.pack(padx=10, pady=(0, 8), anchor="w")

        # Action buttons
        act_row = ctk.CTkFrame(parent, fg_color="transparent")
        act_row.pack(fill="x", padx=10, pady=(4, 10))

        self.btn_send_sign = ctk.CTkButton(
            act_row, text="▶  SEND MESSAGE", height=40,
            fg_color=ACCENT, hover_color="#00b8d4", text_color=BG,
            font=("Courier New", 12, "bold"), corner_radius=8,
            command=self._on_send_signed_message)
        self.btn_send_sign.pack(fill="x", pady=(0, 6))

        del_row = ctk.CTkFrame(act_row, fg_color="transparent")
        del_row.pack(fill="x")
        ctk.CTkButton(del_row, text="DEL", height=32, width=100,
                      fg_color=BTN_H, hover_color="#2a1a1a", text_color=ACCENT2,
                      font=("Courier New", 10, "bold"), corner_radius=6,
                      command=self._sign_delete_last).pack(side="left", padx=(0, 6))
        ctk.CTkButton(del_row, text="CLEAR", height=32, width=100,
                      fg_color=BTN_H, hover_color="#2a1a1a", text_color=MUTED,
                      font=("Courier New", 10, "bold"), corner_radius=6,
                      command=self._sign_clear).pack(side="left")

    # ── Text → Sign panel ────────────────────────────────────────────────────

    def _build_text_to_sign_panel(self, parent):
        ctk.CTkLabel(parent, text="TEXT → SIGN",
                     font=("Courier New", 11, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=14, pady=(10, 0))
        ctk.CTkLabel(parent, text="Hearing user replies here",
                     font=("Courier New", 9), text_color=MUTED).pack(
            anchor="nw", padx=14)

        # Sign display canvas
        ctk.CTkLabel(parent, text="SIGN DISPLAY",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=14, pady=(8, 0))

        self.sign_canvas_frame = ctk.CTkFrame(parent, fg_color="#0a0a0c",
                                               corner_radius=10,
                                               width=CANVAS_W, height=CANVAS_H)
        self.sign_canvas_frame.pack(padx=10, pady=4)
        self.sign_canvas_frame.pack_propagate(False)

        self.sign_display_lbl = ctk.CTkLabel(self.sign_canvas_frame, text="",
                                              font=("Segoe UI Emoji", 64),
                                              text_color=MUTED,
                                              fg_color="#0a0a0c")
        self.sign_display_lbl.pack(expand=True)

        # Progress / label
        info_card = ctk.CTkFrame(parent, fg_color=CARD, corner_radius=10)
        info_card.pack(fill="x", padx=10, pady=4)
        self.sign_word_lbl = ctk.CTkLabel(info_card, text="—",
                                          font=("Courier New", 28, "bold"),
                                          text_color=ACCENT2)
        self.sign_word_lbl.pack(pady=(6, 2))
        self.sign_note_lbl = ctk.CTkLabel(info_card,
                                          text="Enter text below and press Send Reply",
                                          font=("Courier New", 10),
                                          text_color=MUTED)
        self.sign_note_lbl.pack(pady=(0, 6))
        self.sign_prog_bar = ctk.CTkProgressBar(info_card, height=5,
                                                 fg_color="#1e1e28",
                                                 progress_color=ACCENT2,
                                                 corner_radius=3)
        self.sign_prog_bar.set(0)
        self.sign_prog_bar.pack(fill="x", padx=10, pady=(0, 8))

        # Speed
        spd_row = ctk.CTkFrame(parent, fg_color="transparent")
        spd_row.pack(fill="x", padx=10, pady=4)
        ctk.CTkLabel(spd_row, text="SPEED",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            side="left")
        spd_slider = ctk.CTkSlider(spd_row, from_=0.5, to=4.0,
                                   variable=self._speed_var,
                                   fg_color="#1e1e28",
                                   progress_color=ACCENT2,
                                   button_color=ACCENT2, width=140)
        spd_slider.pack(side="left", padx=8)
        self.spd_lbl = ctk.CTkLabel(spd_row, text="1.5×",
                                     font=("Courier New", 10), text_color=MUTED)
        self.spd_lbl.pack(side="left")
        self._speed_var.trace_add("write", lambda *_:
            self.spd_lbl.configure(text=f"{self._speed_var.get():.1f}×"))

        # Input
        ctk.CTkLabel(parent, text="TYPE / SPEAK REPLY",
                     font=("Courier New", 9, "bold"), text_color=MUTED).pack(
            anchor="nw", padx=14, pady=(8, 0))

        inp_frame = ctk.CTkFrame(parent, fg_color="transparent")
        inp_frame.pack(fill="x", padx=10, pady=4)

        self.reply_entry = ctk.CTkEntry(inp_frame,
                                        font=("Courier New", 13),
                                        fg_color=CARD, text_color=TEXT,
                                        placeholder_text="Type your reply…",
                                        border_color=BORDER, corner_radius=8,
                                        height=38)
        self.reply_entry.pack(fill="x", pady=(0, 6))
        self.reply_entry.bind("<Return>", lambda e: self._on_send_reply())

        btn_row2 = ctk.CTkFrame(inp_frame, fg_color="transparent")
        btn_row2.pack(fill="x")

        self.btn_send_reply = ctk.CTkButton(
            btn_row2, text="▶  SEND REPLY", height=38, width=160,
            fg_color=MUTED, hover_color=MUTED, text_color=BG,
            font=("Courier New", 12, "bold"), corner_radius=8,
            command=self._on_send_reply)
        self.btn_send_reply.pack(side="left", padx=(0, 6))

        self.btn_mic = ctk.CTkButton(
            btn_row2, text="🎤 MIC", height=38, width=90,
            fg_color=BTN_H, hover_color="#222230", text_color=GREEN,
            font=("Courier New", 11, "bold"), corner_radius=8,
            command=self._on_speech_input)
        self.btn_mic.pack(side="left")

        # Letter strip (placeholder row kept for layout consistency)
        self.letter_strip = ctk.CTkFrame(parent, fg_color="transparent")
        self.letter_strip.pack(fill="x", padx=10, pady=4)

    # ── Chat log panel ───────────────────────────────────────────────────────

    def _build_chat_log_panel(self, parent):
        ctk.CTkLabel(parent, text="CONVERSATION",
                     font=("Courier New", 10, "bold"), text_color=MUTED).pack(
            padx=12, pady=(12, 4), anchor="nw")

        self.chat_scroll = ctk.CTkScrollableFrame(parent, fg_color=BG,
                                                   corner_radius=8)
        self.chat_scroll.pack(fill="both", expand=True, padx=8, pady=(0, 8))

    def _add_chat_bubble(self, role, text):
        """Add a bubble to the chat log."""
        self.chat_log.append({"role": role, "text": text})
        is_deaf = role == "deaf"
        bg  = "#0d2a3a" if is_deaf else "#1a1a0d"
        tc  = ACCENT    if is_deaf else ACCENT2
        tag = "Deaf"    if is_deaf else "Hearing"

        bubble = ctk.CTkFrame(self.chat_scroll, fg_color=bg, corner_radius=10)
        bubble.pack(fill="x", pady=4, padx=4,
                    anchor="w" if is_deaf else "e")

        ctk.CTkLabel(bubble, text=tag,
                     font=("Courier New", 8, "bold"), text_color=tc).pack(
            anchor="nw", padx=8, pady=(6, 0))
        ctk.CTkLabel(bubble, text=text,
                     font=("Courier New", 11), text_color=TEXT,
                     wraplength=170, justify="left").pack(padx=8, pady=(2, 8))

        # Auto-scroll to bottom
        self.root.after(50, lambda: self.chat_scroll._parent_canvas.yview_moveto(1.0))

    # ══════════════════════════════════════════════════════════════════════════
    #  CONVERSATION MODE MANAGEMENT
    # ══════════════════════════════════════════════════════════════════════════

    def _activate_mode(self, mode):
        """Switch visual state to 'sign_to_text' or 'text_to_sign'."""
        self.conv_mode = mode
        if mode == "sign_to_text":
            self.col_a.configure(border_color=ACCENT,  border_width=2)
            self.col_b.configure(border_color=MUTED,   border_width=2)
            self.mode_badge.configure(
                text="● SIGN → TEXT  (Deaf Turn)", text_color=ACCENT)
            self.btn_send_sign.configure(fg_color=ACCENT,
                                          hover_color="#00b8d4",
                                          text_color=BG, state="normal")
            self.btn_send_reply.configure(fg_color=MUTED,
                                           hover_color=MUTED,
                                           text_color=BG, state="disabled")
            self.btn_mic.configure(state="disabled")
        else:
            self.col_a.configure(border_color=MUTED,   border_width=2)
            self.col_b.configure(border_color=ACCENT2, border_width=2)
            self.mode_badge.configure(
                text="● TEXT → SIGN  (Hearing Turn)", text_color=ACCENT2)
            self.btn_send_sign.configure(fg_color=BTN_H,
                                          hover_color=BTN_H,
                                          text_color=MUTED, state="disabled")
            self.btn_send_reply.configure(fg_color=ACCENT2,
                                           hover_color="#cc2a55",
                                           text_color=BG, state="normal")
            self.btn_mic.configure(state="normal")

    # ══════════════════════════════════════════════════════════════════════════
    #  FLOW:  SIGN → TEXT
    # ══════════════════════════════════════════════════════════════════════════

    def _on_send_signed_message(self):
        text = self.sign_buffer.strip()
        if not text:
            return
        ConfirmDialog(
            self.root,
            title="Send Signed Message",
            message=f'Signed message:\n\n"{text}"\n\nHow would you like to send it?',
            buttons=[
                ("confirm", "Show Text"),
                ("tts",     "Speak Aloud"),
                ("cancel",  "Cancel"),
            ],
            on_result=lambda choice: self._after_sign_confirm(choice, text),
            accent=ACCENT,
        )

    def _after_sign_confirm(self, choice, text):
        if choice == "cancel":
            return
        if choice == "tts":
            threading.Thread(target=self._tts_run, args=(text,),
                             daemon=True).start()
        self._add_chat_bubble("deaf", text)
        self.sign_buffer = ""
        self._update_sign_buffer_ui()
        self.root.after(300, self._ask_switch_to_text_to_sign)

    def _ask_switch_to_text_to_sign(self):
        ConfirmDialog(
            self.root,
            title="Switch Mode",
            message="Message sent!\n\nSwitch to TEXT → SIGN so the hearing person can reply?",
            buttons=[
                ("switch", "⇄  Switch"),
                ("cancel", "Stay Here"),
            ],
            on_result=lambda c: self._activate_mode("text_to_sign") if c == "switch" else None,
            accent=WARN_FG,
        )

    # ══════════════════════════════════════════════════════════════════════════
    #  FLOW:  TEXT → SIGN
    # ══════════════════════════════════════════════════════════════════════════

    def _on_send_reply(self):
        text = self.reply_entry.get().strip()
        if not text:
            return
        ConfirmDialog(
            self.root,
            title="Send Reply as Signs",
            message=f'Reply text:\n\n"{text}"\n\nPlay this as ASL signs for the deaf user?',
            buttons=[
                ("confirm", " Play Signs"),
                ("cancel",  " Cancel"),
            ],
            on_result=lambda choice: self._after_reply_confirm(choice, text),
            accent=ACCENT2,
        )

    def _after_reply_confirm(self, choice, text):
        if choice == "cancel":
            return
        self._add_chat_bubble("hearing", text)
        self.reply_entry.delete(0, "end")
        self.sign_word_lbl.configure(text="⏳")
        self.sign_note_lbl.configure(text="Loading signs from local assets…")
        # build_sign_sequence is fast (all local) but run in a thread to keep
        # the UI responsive when opening many large GIF files.
        threading.Thread(target=self._load_and_play_signs,
                         args=(text,), daemon=True).start()

    def _load_and_play_signs(self, text):
        seq = build_sign_sequence(text)
        self.root.after(0, lambda: self._start_sign_playback(seq))

    def _start_sign_playback(self, seq):
        self._stop_sign_playback()
        self.sign_sequence = seq
        self.sign_seq_idx  = 0
        self.sign_playing  = True
        self.sign_prog_bar.set(0)
        self._advance_sign()

    def _advance_sign(self):
        if not self.sign_playing:
            return
        if self.sign_seq_idx >= len(self.sign_sequence):
            self.sign_playing = False
            self.sign_word_lbl.configure(text="✓")
            self.sign_note_lbl.configure(text="Done!")
            self.sign_prog_bar.set(1.0)
            self.root.after(600, self._ask_switch_to_sign_to_text)
            return

        item = self.sign_sequence[self.sign_seq_idx]
        pct  = (self.sign_seq_idx + 1) / max(len(self.sign_sequence), 1)
        self.sign_prog_bar.set(pct)

        if item["kind"] == "space":
            self.sign_seq_idx += 1
            self.sign_gif_job = self.root.after(
                int(250 / self._speed_var.get()), self._advance_sign)
            return

        self.sign_word_lbl.configure(text=item["label"])
        kind = item["kind"]
        note = "Word sign (GIF)" if kind == "gif" else "Fingerspelling"
        self.sign_note_lbl.configure(text=f"{note}: {item['label']}")

        frames = item.get("frames", [])
        if not frames:
            self.sign_seq_idx += 1
            self.sign_gif_job = self.root.after(200, self._advance_sign)
            return

        if kind == "gif" and len(frames) > 1:
            self._play_gif_frames(frames, 0)
        else:
            self._show_sign_frame(frames[0])
            hold_ms = int(900 / self._speed_var.get())
            self.sign_seq_idx += 1
            self.sign_gif_job = self.root.after(hold_ms, self._advance_sign)

    def _play_gif_frames(self, frames, frame_idx):
        if not self.sign_playing:
            return
        self._show_sign_frame(frames[frame_idx])
        nxt = frame_idx + 1
        if nxt < len(frames):
            delay = max(40, int(60 / self._speed_var.get()))
            self.sign_gif_job = self.root.after(
                delay, lambda: self._play_gif_frames(frames, nxt))
        else:
            self.sign_seq_idx += 1
            self.sign_gif_job = self.root.after(300, self._advance_sign)

    def _show_sign_frame(self, img: Image.Image):
        ph = ImageTk.PhotoImage(img)
        self._static_img_ref = ph
        self.sign_display_lbl.configure(image=ph, text="")
        self.sign_display_lbl._img = ph

    def _stop_sign_playback(self):
        self.sign_playing = False
        if self.sign_gif_job:
            self.root.after_cancel(self.sign_gif_job)
            self.sign_gif_job = None

    def _ask_switch_to_sign_to_text(self):
        ConfirmDialog(
            self.root,
            title="Switch Mode",
            message="Signs played!\n\nSwitch back to SIGN → TEXT so the deaf person can reply?",
            buttons=[
                ("switch", "⇄  Switch"),
                ("cancel", "Stay Here"),
            ],
            on_result=lambda c: self._activate_mode("sign_to_text") if c == "switch" else None,
            accent=WARN_FG,
        )

    # ══════════════════════════════════════════════════════════════════════════
    #  SPEECH INPUT  (for hearing user)
    # ══════════════════════════════════════════════════════════════════════════

    def _on_speech_input(self):
        self.btn_mic.configure(text="  Listening…", text_color="#ff3d71")
        threading.Thread(target=self._listen_mic, daemon=True).start()

    def _listen_mic(self):
        r = sr.Recognizer()
        try:
            with sr.Microphone() as src:
                r.adjust_for_ambient_noise(src, duration=0.5)
                audio = r.listen(src, timeout=6, phrase_time_limit=8)
            text = r.recognize_google(audio)
            self.root.after(0, lambda: self.reply_entry.delete(0, "end"))
            self.root.after(0, lambda: self.reply_entry.insert(0, text))
        except sr.WaitTimeoutError:
            pass
        except sr.UnknownValueError:
            pass
        except Exception:
            pass
        finally:
            self.root.after(0, lambda: self.btn_mic.configure(
                text="🎤 MIC", text_color=GREEN))

    # ══════════════════════════════════════════════════════════════════════════
    #  MODEL LOADING
    # ══════════════════════════════════════════════════════════════════════════

    def _load_models_async(self):
        threading.Thread(target=self._load_models, daemon=True).start()

    def _load_models(self):
        try:
            self.letter_model = tf.keras.models.load_model(MODEL_LETTER)
            self.root.after(0, lambda: self.status_dot.configure(
                text="● LETTER ✓  WORD…", text_color="#ffaa00"))
        except Exception as e:
            self.root.after(0, lambda: self.status_dot.configure(
                text=f"⚠ LETTER: {e}", text_color=ACCENT2))

        try:
            self.word_model = tf.keras.models.load_model(MODEL_WORD)
            self.root.after(0, lambda: self.status_dot.configure(
                text="● READY", text_color=ACCENT))
        except Exception as e:
            self.root.after(0, lambda: self.status_dot.configure(
                text=f"⚠ WORD: {e}", text_color=ACCENT2))

        self.holistic = mp_holistic.Holistic(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5)
        self.hand_detector = mp_hands_mod.Hands(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
            max_num_hands=1)

        self.root.after(0, lambda: self._activate_mode("sign_to_text"))

    # ══════════════════════════════════════════════════════════════════════════
    #  CAMERA LOOP
    # ══════════════════════════════════════════════════════════════════════════

    def _camera_loop(self):
        if not self.running:
            return
        ret, frame = self.cap.read()
        if ret:
            frame = cv2.flip(frame, 1)
            skel  = np.zeros((224, 224, 3), dtype=np.uint8)
            rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            if self.sign_mode == "letter":
                self._process_letter(rgb, skel)
            else:
                self._process_word(rgb)

            # Draw overlay badge
            h, w = frame.shape[:2]
            tag  = "LETTER" if self.sign_mode == "letter" else "WORD"
            col  = (0, 229, 255) if self.sign_mode == "letter" else (255, 61, 113)
            cv2.rectangle(frame, (w - 110, 0), (w, 28), (13, 13, 15), -1)
            cv2.putText(frame, tag, (w - 100, 20),
                        cv2.FONT_HERSHEY_DUPLEX, 0.65, col, 1, cv2.LINE_AA)

            # Video display
            scale = 400 / frame.shape[1]
            disp  = cv2.resize(rgb, (400, int(frame.shape[0] * scale)))
            img   = Image.fromarray(disp)
            cimg  = ctk.CTkImage(light_image=img, dark_image=img,
                                  size=(400, int(frame.shape[0] * scale)))
            self.video_lbl.configure(image=cimg)
            self.video_lbl._image = cimg

            # Skeleton
            skel_rgb = cv2.cvtColor(self.skeleton_frame, cv2.COLOR_BGR2RGB)
            skel_img = Image.fromarray(skel_rgb)
            cskel    = ctk.CTkImage(light_image=skel_img, dark_image=skel_img,
                                     size=(120, 120))
            self.skel_lbl.configure(image=cskel)
            self.skel_lbl._image = cskel

        self.root.after(15, self._camera_loop)

    # ══════════════════════════════════════════════════════════════════════════
    #  SIGN DETECTION  (letter + word)
    # ══════════════════════════════════════════════════════════════════════════

    def _process_letter(self, rgb, skel):
        if self.hand_detector is None or self.letter_model is None:
            return
        result = self.hand_detector.process(rgb)
        if result.multi_hand_landmarks:
            for hand in result.multi_hand_landmarks:
                pts = normalize_landmarks(hand.landmark, 224, 224)
                draw_hand_skeleton(skel, pts)
            self.skeleton_frame = skel
            inp  = tf.keras.applications.resnet50.preprocess_input(
                np.expand_dims(skel, axis=0).astype(np.float32))
            pred = self.letter_model.predict(inp, verbose=0)[0]
            idx  = int(np.argmax(pred)); conf = float(pred[idx])
            label = self.LETTER_LABELS[idx]
            self.cur_prediction = label; self.cur_confidence = conf
            self._update_pred_ui(label, conf)
            if conf >= self.CONF_THRESH and label not in ('ok',):
                now = time.time()
                if self.pending_letter != label:
                    self.pending_letter = label; self.pending_since = now
                elapsed  = now - (self.pending_since or now)
                progress = min(elapsed / self.DEBOUNCE_SEC, 1.0)
                self.debounce_bar.set(progress)
                self.debounce_lbl.configure(
                    text=f'confirming "{label}"… {int(progress * 100)}%')
                if elapsed >= self.DEBOUNCE_SEC:
                    self._confirm_letter(label)
                    self.pending_letter = None; self.pending_since = None
                    self.debounce_bar.set(0)
                    self.debounce_lbl.configure(text="hold sign to confirm…")
            else:
                self.pending_letter = None; self.pending_since = None
                self.debounce_bar.set(0)

    def _confirm_letter(self, letter):
        if letter == 'del':
            self.sign_buffer = self.sign_buffer[:-1]
        elif letter == 'space':
            self.sign_buffer += ' '
        else:
            self.sign_buffer += letter
        self._update_sign_buffer_ui()

    def _process_word(self, rgb):
        if self.holistic is None or self.word_model is None:
            return
        img_rgb = rgb.copy(); img_rgb.flags.writeable = False
        result  = self.holistic.process(img_rgb)
        kp = extract_keypoints(result)
        self.word_sequence.append(kp)
        if len(self.word_sequence) == 30:
            seq  = np.expand_dims(np.array(self.word_sequence), axis=0)
            pred = self.word_model.predict(seq, verbose=0)[0]
            idx  = int(np.argmax(pred)); conf = float(pred[idx])
            label = self.WORD_LABELS[idx]
            self._update_pred_ui(label, conf)
            if conf >= self.CONF_THRESH:
                if idx == self.last_word_pred:
                    self.word_stab_count += 1
                else:
                    self.last_word_pred = idx; self.word_stab_count = 1
                if (self.word_stab_count >= self.WORD_STAB_FRM
                        and label != 'nothing'
                        and label != self.last_confirmed_word):
                    self.sign_buffer += label + ' '
                    self.last_confirmed_word = label
                    self.word_stab_count = 0
                    self._update_sign_buffer_ui()
            else:
                self.last_word_pred = None; self.word_stab_count = 0

    # ══════════════════════════════════════════════════════════════════════════
    #  SIGN MODE TOGGLE
    # ══════════════════════════════════════════════════════════════════════════

    def _set_sign_mode(self, mode):
        self.sign_mode = mode
        if mode == "letter":
            self.btn_letter_mode.configure(fg_color=ACCENT, text_color=BG)
            self.btn_word_mode.configure(fg_color=BTN_H,   text_color=MUTED)
        else:
            self.btn_word_mode.configure(fg_color=ACCENT,  text_color=BG)
            self.btn_letter_mode.configure(fg_color=BTN_H, text_color=MUTED)
        self.pending_letter = None; self.pending_since = None
        self.word_sequence.clear(); self.word_stab_count = 0
        self.last_word_pred = None; self.last_confirmed_word = None

    # ══════════════════════════════════════════════════════════════════════════
    #  UI UPDATE HELPERS
    # ══════════════════════════════════════════════════════════════════════════

    def _update_pred_ui(self, label, conf):
        self.pred_lbl.configure(text=label.upper())
        self.conf_bar.set(conf)
        self.conf_lbl.configure(text=f"{int(conf * 100)}%")

    def _update_sign_buffer_ui(self):
        self.sign_buffer_lbl.configure(
            text=self.sign_buffer if self.sign_buffer else "")

    def _sign_delete_last(self):
        self.sign_buffer = self.sign_buffer.rstrip()
        if ' ' in self.sign_buffer:
            self.sign_buffer = self.sign_buffer.rsplit(' ', 1)[0] + ' '
        else:
            self.sign_buffer = self.sign_buffer[:-1]
        self._update_sign_buffer_ui()

    def _sign_clear(self):
        self.sign_buffer = ""
        self._update_sign_buffer_ui()
        self.pending_letter = None; self.pending_since = None
        self.debounce_bar.set(0)

    def _tts_run(self, text):
        self.tts_engine.say(text)
        self.tts_engine.runAndWait()

    # ══════════════════════════════════════════════════════════════════════════
    #  CLOSE
    # ══════════════════════════════════════════════════════════════════════════

    def on_close(self):
        self.running = False
        self._stop_sign_playback()
        self.cap.release()
        if self.holistic:
            self.holistic.close()
        if self.hand_detector:
            self.hand_detector.close()
        self.root.destroy()


# ──────────────────────────────────────────────────────────────────────────────
#  ENTRY POINT
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    ctk.set_appearance_mode("dark")
    ctk.set_default_color_theme("blue")

    root = ctk.CTk()
    app  = ASLBidirectionalApp(root)
    root.protocol("WM_DELETE_WINDOW", app.on_close)
    root.mainloop()

bgerror failed to handle background error.
    Original error: invalid command name "140545624719616update"
    Error in bgerror: can't invoke "tk" command: application has been destroyed
bgerror failed to handle background error.
    Original error: invalid command name "140548199433408_camera_loop"
    Error in bgerror: can't invoke "tk" command: application has been destroyed
bgerror failed to handle background error.
    Original error: invalid command name "140545624333440check_dpi_scaling"
    Error in bgerror: can't invoke "tk" command: application has been destroyed
I0000 00:00:1776068348.735390  182607 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776068348.738818  183454 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.4), renderer: Mesa Intel(R) UHD Graphics (TGL GT1)
I0000 00:00:1776068348.765362  182607 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776068348.768558  183469 gl_context.cc: